In [ ]:
from pathlib import Path

dataset_folder = Path(
    "/kaggle/input/competitions/"
    "enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = dataset_folder / "train.parquet"
test_path = dataset_folder / "test.parquet"
submission_path = dataset_folder / "sample_submission.csv"

print("Train file exists:", train_path.exists())
print("Test file exists:", test_path.exists())
print("Sample submission exists:", submission_path.exists())

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

test_file = pq.ParquetFile(test_path)
sample_submission = pd.read_csv(submission_path)

print("Test spectra:", test_file.metadata.num_rows)
print("Test columns:", test_file.schema_arrow.names)
print("Submission columns:", sample_submission.columns.tolist())
print("Sample submission rows:", len(sample_submission))

In [ ]:
# CASMI 2026 - Prepare test spectra for exact matching

import hashlib
import numpy as np
from collections import defaultdict

test_columns = [
    "molecule_id",
    "spectrum_id",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

test_spectra = test_file.read(
    columns=test_columns
).to_pandas()


def spectrum_fingerprint(row):
    mz = np.asarray(row.ms2_mzs, dtype=np.float64)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=np.float64
    )

    fingerprint = hashlib.blake2b(
        mz.tobytes() + intensity.tobytes(),
        digest_size=16
    ).digest()

    return (row.adduct, len(mz), fingerprint)


# Connect each fingerprint to its test molecule and spectrum
test_lookup = defaultdict(list)

for row in test_spectra.itertuples(index=False):
    fingerprint = spectrum_fingerprint(row)

    test_lookup[fingerprint].append(
        (row.molecule_id, row.spectrum_id)
    )


print("CASMI 2026 - Test Spectra Prepared")
print("----------------------------------")
print("Test spectra:", len(test_spectra))
print(
    "Unique test molecules:",
    test_spectra["molecule_id"].nunique()
)
print("Unique spectrum fingerprints:", len(test_lookup))

print(
    "Submission molecule IDs match test data:",
    set(sample_submission["molecule_id"])
    == set(test_spectra["molecule_id"])
)

print("\nTest spectra prepared successfully!")

In [ ]:
# CASMI 2026 - Find exact matches in the training dataset

import pyarrow.parquet as pq
from collections import defaultdict

train_file = pq.ParquetFile(train_path)

train_columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Skip spectra with adduct/peak-count combinations
# that do not occur in the test data.
test_combinations = {
    (row.adduct, len(row.ms2_mzs))
    for row in test_spectra.itertuples(index=False)
}

# molecule_id -> (inchikey14, SMILES) -> matched spectrum IDs
exact_hits = defaultdict(lambda: defaultdict(set))

print("CASMI 2026 - Exact Match Search")
print("-------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=10000,
        columns=train_columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            combination = (
                row.adduct,
                len(row.ms2_mzs)
            )

            if combination not in test_combinations:
                continue

            fingerprint = spectrum_fingerprint(row)

            for molecule_id, spectrum_id in test_lookup.get(
                fingerprint, []
            ):
                exact_hits[molecule_id][
                    (row.inchikey14, row.normalized_smiles)
                ].add(spectrum_id)

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Summarize without assuming that all test molecules matched
test_molecule_count = test_spectra["molecule_id"].nunique()

matched_molecules = len(exact_hits)

unique_matches = sum(
    len(structures) == 1
    for structures in exact_hits.values()
)

ambiguous_matches = sum(
    len(structures) > 1
    for structures in exact_hits.values()
)

print("\nExact matching results:")
print("Test molecules:", test_molecule_count)
print("Molecules with exact matches:", matched_molecules)
print("Molecules with one matched structure:", unique_matches)
print("Molecules with multiple matched structures:", ambiguous_matches)
print(
    "Molecules without an exact match:",
    test_molecule_count - matched_molecules
)

In [ ]:
# CASMI 2026 - Prepare exact-match predictions

exact_predictions = {}
unresolved_molecules = []

for molecule_id in sample_submission["molecule_id"]:

    matched_structures = exact_hits.get(molecule_id, {})

    # Use an exact match only when it identifies one structure
    if len(matched_structures) == 1:

        inchikey14, smiles = next(
            iter(matched_structures.keys())
        )

        if (
            isinstance(smiles, str)
            and smiles.strip()
            and ";" not in smiles
        ):
            exact_predictions[molecule_id] = smiles
        else:
            unresolved_molecules.append(molecule_id)

    else:
        unresolved_molecules.append(molecule_id)


print("CASMI 2026 - Exact Predictions")
print("------------------------------")
print("Submission molecules:", len(sample_submission))
print("Exact predictions:", len(exact_predictions))
print("Unresolved molecules:", len(unresolved_molecules))

print("\nFirst 3 exact predictions:")
for molecule_id, smiles in list(exact_predictions.items())[:3]:
    print(molecule_id, "->", smiles)

print("\nExact prediction mapping prepared!")

In [ ]:
# CASMI 2026 - Build a fallback candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

for batch in train_file.iter_batches(
    batch_size=20000,
    columns=library_columns
):
    batch_df = batch.to_pandas()

    # Remove incomplete rows and repeated molecular structures
    batch_df = batch_df.dropna(
        subset=library_columns
    ).drop_duplicates(subset="inchikey14")

    new_rows = batch_df[
        ~batch_df["inchikey14"].isin(seen_keys)
    ]

    if not new_rows.empty:
        library_parts.append(new_rows)
        seen_keys.update(new_rows["inchikey14"])

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

print("CASMI 2026 - Fallback Candidate Library")
print("---------------------------------------")
print("Unique molecular structures:", len(full_candidate_library))
print(
    "Missing SMILES:",
    full_candidate_library["normalized_smiles"].isna().sum()
)
print(
    "Missing molecular formulas:",
    full_candidate_library["molecular_formula"].isna().sum()
)

print("\nCandidate library prepared!")

In [ ]:
# CASMI 2026 - Calculate candidate reference masses

import re
import numpy as np

# Monoisotopic atomic masses in Da
atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}


def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    # Remove trailing charge notation, where present
    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    # Reject formulas we cannot parse completely
    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )


full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

print("CASMI 2026 - Reference Mass Calculation")
print("---------------------------------------")
print("Total candidates:", len(full_candidate_library))
print(
    "Candidates with reference mass:",
    int(full_candidate_library["reference_mass"].notna().sum())
)
print(
    "Candidates with missing reference mass:",
    int(full_candidate_library["reference_mass"].isna().sum())
)

print("\nReference mass calculation completed!")

In [ ]:
# CASMI 2026 - Calculate neutral masses for test molecules

import pandas as pd

adduct_shifts = {
    "[M+H]+": 1.007276466621,
    "[M-H]-": -1.007276466621,
    "[M+CH2O2-H]-": 44.99820284,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969401
}

# Read the test metadata available during this notebook run
mass_data = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

mass_data["adduct_shift"] = (
    mass_data["adduct"].map(adduct_shifts)
)

mass_data["neutral_mass"] = (
    pd.to_numeric(mass_data["precursor_mz"], errors="coerce")
    - mass_data["adduct_shift"]
)

# Calculate one median neutral mass per molecule
test_molecules = (
    mass_data.groupby("molecule_id", as_index=False)
    .agg(
        median_neutral_mass=("neutral_mass", "median"),
        num_spectra=("spectrum_id", "count")
    )
)

print("CASMI 2026 - Test Neutral Masses")
print("--------------------------------")
print("Test spectra:", len(mass_data))
print("Test molecules:", len(test_molecules))

print(
    "Spectra with unknown adducts:",
    int(mass_data["adduct_shift"].isna().sum())
)

print(
    "Molecules with missing neutral mass:",
    int(test_molecules["median_neutral_mass"].isna().sum())
)

print("\nFirst 5 molecules:")
print(test_molecules.head().to_string(index=False))

In [ ]:
# CASMI 2026 - Generate submission.csv

import numpy as np
import pandas as pd
from pathlib import Path

# Prepare the mass-based candidate library
mass_library = (
    full_candidate_library
    .dropna(subset=["reference_mass", "normalized_smiles"])
    .sort_values("reference_mass")
    .reset_index(drop=True)
)

sorted_masses = mass_library["reference_mass"].to_numpy(
    dtype=float
)
sorted_smiles = mass_library["normalized_smiles"].to_numpy()

mass_lookup = test_molecules.set_index(
    "molecule_id"
)["median_neutral_mass"]

submission_rows = []

for molecule_id in sample_submission["molecule_id"]:

    selected_smiles = []
    seen_smiles = set()

    # Place an unambiguous exact match first, if available
    exact_smiles = exact_predictions.get(molecule_id)

    if exact_smiles is not None:
        selected_smiles.append(exact_smiles)
        seen_smiles.add(exact_smiles)

    # Fill remaining positions using nearby reference masses
    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        raise ValueError(
            f"No usable neutral mass for {molecule_id}"
        )

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(
                sorted_masses[left] - query_mass
            )
            right_error = abs(
                sorted_masses[right] - query_mass
            )

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(selected_smiles)
    })

submission = pd.DataFrame(submission_rows)

# Check the required format
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert len(submission) == len(sample_submission)

assert (
    submission["molecule_id"].tolist()
    == sample_submission["molecule_id"].tolist()
)

assert all(
    len(smiles.split(";")) == 25
    and len(set(smiles.split(";"))) == 25
    for smiles in submission["smiles"]
)

# Save using the filename required by Kaggle
output_path = Path("/kaggle/working/submission.csv")
submission.to_csv(output_path, index=False)

print("CASMI 2026 - Submission Created")
print("--------------------------------")
print("File exists:", output_path.exists())
print("File path:", output_path)
print("Submission rows:", len(submission))
print("SMILES per molecule: 25")
print(
    "Molecules with exact prediction:",
    sum(
        molecule_id in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print(
    "Molecules using mass-only fallback:",
    sum(
        molecule_id not in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print("\nSubmission format validation passed!")

In [10]:
# CASMI 2026 - Evaluate the current mass-only fallback

import numpy as np

# Use the variables already created in our submission notebook:
# sorted_masses, sorted_smiles, mass_lookup, exact_predictions

total = 0
top1_matches = 0
top25_matches = 0
reciprocal_rank_sum = 0.0

for molecule_id in sample_submission["molecule_id"]:

    known_smiles = exact_predictions.get(molecule_id)

    if known_smiles is None:
        continue

    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        continue

    # Recreate the mass-only ranking without using the exact match
    selected_smiles = []
    seen_smiles = set()

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(sorted_masses[left] - query_mass)
            right_error = abs(sorted_masses[right] - query_mass)

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    total += 1

    if known_smiles in selected_smiles:
        rank = selected_smiles.index(known_smiles) + 1
        top25_matches += 1
        reciprocal_rank_sum += 1.0 / rank

        if rank == 1:
            top1_matches += 1

print("CASMI 2026 - Mass-Only Fallback Diagnostic")
print("------------------------------------------")
print("Molecules evaluated:", total)
print("Exact-match SMILES at rank 1:", top1_matches)
print("Exact-match SMILES within top 25:", top25_matches)

if total:
    print("Top-1 rate:", round(top1_matches / total, 4))
    print("Top-25 rate:", round(top25_matches / total, 4))
    print("Mean Reciprocal Rank @ 25:", round(
        reciprocal_rank_sum / total, 4
    ))

CASMI 2026 - Mass-Only Fallback Diagnostic
------------------------------------------
Molecules evaluated: 400
Exact-match SMILES at rank 1: 23
Exact-match SMILES within top 25: 221
Top-1 rate: 0.0575
Top-25 rate: 0.5525
Mean Reciprocal Rank @ 25: 0.1386


In [11]:
# CASMI 2026 - Diagnose mass-only fallback misses

import numpy as np

# Find reference masses associated with each SMILES
masses_by_smiles = {}

for row in full_candidate_library[
    ["normalized_smiles", "reference_mass"]
].itertuples(index=False):

    if np.isfinite(row.reference_mass):
        masses_by_smiles.setdefault(
            row.normalized_smiles, []
        ).append(float(row.reference_mass))


diagnostic = {
    "correct_smiles_not_in_library": 0,
    "correct_mass_within_20_ppm": 0,
    "correct_mass_above_20_ppm": 0,
    "missed_top25_but_mass_within_20_ppm": 0
}

examples = []

for molecule_id in sample_submission["molecule_id"]:

    true_smiles = exact_predictions[molecule_id]
    query_mass = float(mass_lookup.loc[molecule_id])

    reference_masses = masses_by_smiles.get(true_smiles)

    if not reference_masses:
        diagnostic["correct_smiles_not_in_library"] += 1
        continue

    best_mass_error_ppm = min(
        abs(reference_mass - query_mass)
        / query_mass * 1_000_000
        for reference_mass in reference_masses
    )

    if best_mass_error_ppm <= 20:
        diagnostic["correct_mass_within_20_ppm"] += 1
    else:
        diagnostic["correct_mass_above_20_ppm"] += 1

    # Is the correct SMILES in our previous mass-only top 25?
    predicted_smiles = submission.loc[
        submission["molecule_id"] == molecule_id,
        "smiles"
    ].iloc[0].split(";")

    # Remove the exact-match prediction from consideration.
    # The remaining entries are not necessarily identical to
    # the mass-only top 25, so use the mass-only search below.
    left = int(np.searchsorted(sorted_masses, query_mass)) - 1
    right = left + 1
    mass_only_top25 = []
    seen = set()

    while len(mass_only_top25) < 25:
        if left < 0:
            index = right
            right += 1
        elif right >= len(sorted_masses):
            index = left
            left -= 1
        elif abs(sorted_masses[left] - query_mass) <= abs(
            sorted_masses[right] - query_mass
        ):
            index = left
            left -= 1
        else:
            index = right
            right += 1

        smiles = sorted_smiles[index]

        if smiles not in seen:
            mass_only_top25.append(smiles)
            seen.add(smiles)

    if (
        true_smiles not in mass_only_top25
        and best_mass_error_ppm <= 20
    ):
        diagnostic["missed_top25_but_mass_within_20_ppm"] += 1

        if len(examples) < 5:
            examples.append(
                (molecule_id, round(best_mass_error_ppm, 4))
            )


print("CASMI 2026 - Mass-Only Diagnostic")
print("--------------------------------")
for label, value in diagnostic.items():
    print(f"{label}: {value}")

print("\nExamples missed from top 25 despite mass match:")
for molecule_id, error_ppm in examples:
    print(molecule_id, "| mass error:", error_ppm, "ppm")

CASMI 2026 - Mass-Only Diagnostic
--------------------------------
correct_smiles_not_in_library: 0
correct_mass_within_20_ppm: 400
correct_mass_above_20_ppm: 0
missed_top25_but_mass_within_20_ppm: 179

Examples missed from top 25 despite mass match:
m_006153 | mass error: 1.2033 ppm
m_00b5aa | mass error: 2.2235 ppm
m_0259d4 | mass error: 2.0597 ppm
m_02d188 | mass error: 0.6344 ppm
m_050bf1 | mass error: 2.161 ppm


In [12]:
# CASMI 2026 - Prepare references without exact test-spectrum copies

import numpy as np
import pandas as pd

molecule_id = "m_006153"
ppm_tolerance = 20

# Test spectra and observed adducts for this molecule
queries = test_spectra[
    test_spectra["molecule_id"] == molecule_id
].copy()

test_adducts = set(queries["adduct"])

# Identify the structure found through exact matching
true_key, true_smiles = next(
    iter(exact_hits[molecule_id].keys())
)

# Find all mass-matched candidate structures
query_mass = float(
    test_molecules.loc[
        test_molecules["molecule_id"] == molecule_id,
        "median_neutral_mass"
    ].iloc[0]
)

ppm_errors = (
    abs(full_candidate_library["reference_mass"] - query_mass)
    / query_mass
) * 1_000_000

candidate_keys = set(
    full_candidate_library.loc[
        ppm_errors <= ppm_tolerance,
        "inchikey14"
    ]
)

# Fingerprints of this molecule's test spectra
query_fingerprints = {
    spectrum_fingerprint(row)
    for row in queries.itertuples(index=False)
}

reference_parts = []
excluded_exact_copies = 0

# Read training data in batches
for batch in train_file.iter_batches(
    batch_size=20000,
    columns=[
        "inchikey14",
        "adduct",
        "ms2_mzs",
        "ms2_normalized_intensities"
    ]
):
    batch_df = batch.to_pandas()

    matches = batch_df[
        batch_df["inchikey14"].isin(candidate_keys)
        & batch_df["adduct"].isin(test_adducts)
    ].copy()

    if matches.empty:
        continue

    # Exclude reference spectra identical to any query spectrum
    is_exact_copy = matches.apply(
        lambda row: spectrum_fingerprint(row)
        in query_fingerprints,
        axis=1
    )

    excluded_exact_copies += int(is_exact_copy.sum())

    remaining = matches.loc[~is_exact_copy]

    if not remaining.empty:
        reference_parts.append(remaining)

experiment_references = (
    pd.concat(reference_parts, ignore_index=True)
    if reference_parts
    else pd.DataFrame()
)

print("CASMI 2026 - Spectral Ranking Experiment")
print("----------------------------------------")
print("Test molecule:", molecule_id)
print("Test spectra:", len(queries))
print("Mass-matched candidates:", len(candidate_keys))
print("Exact test-spectrum copies excluded:", excluded_exact_copies)
print("Remaining reference spectra:", len(experiment_references))

if not experiment_references.empty:
    print(
        "Candidates with remaining references:",
        experiment_references["inchikey14"].nunique()
    )
    print(
        "Remaining references for the known structure:",
        int(
            (
                experiment_references["inchikey14"]
                == true_key
            ).sum()
        )
    )

CASMI 2026 - Spectral Ranking Experiment
----------------------------------------
Test molecule: m_006153
Test spectra: 6
Mass-matched candidates: 257
Exact test-spectrum copies excluded: 6
Remaining reference spectra: 1376
Candidates with remaining references: 250
Remaining references for the known structure: 1


In [13]:
# CASMI 2026 - Rank candidates without exact spectrum copies

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Convert a spectrum into 1 Da intensity bins
def extract_features(row):
    features = np.zeros(1000, dtype=np.float32)

    mzs = np.asarray(row["ms2_mzs"], dtype=float)
    intensities = np.asarray(
        row["ms2_normalized_intensities"],
        dtype=float
    )

    for mz, intensity in zip(mzs, intensities):
        if np.isfinite(mz) and np.isfinite(intensity):
            index = int(mz)
            if 0 <= index < 1000:
                features[index] = max(
                    features[index], intensity
                )

    return features


# Prepare all mass-matched candidates
ranking = full_candidate_library[
    full_candidate_library["inchikey14"].isin(candidate_keys)
].copy()

ranking["mass_error_ppm"] = (
    abs(ranking["reference_mass"] - query_mass)
    / query_mass * 1_000_000
)

# Give every candidate a score for every query spectrum.
# A missing matching-adduct reference contributes zero.
query_count = len(queries)

score_sums = {
    key: 0.0 for key in candidate_keys
}

matched_query_counts = {
    key: 0 for key in candidate_keys
}

# Compare only spectra with the same adduct
for adduct, query_group in queries.groupby("adduct"):

    ref_group = experiment_references[
        experiment_references["adduct"] == adduct
    ]

    if ref_group.empty:
        continue

    X_query = np.stack(
        query_group.apply(
            extract_features, axis=1
        ).to_numpy()
    )

    X_ref = np.stack(
        ref_group.apply(
            extract_features, axis=1
        ).to_numpy()
    )

    similarity = cosine_similarity(X_query, X_ref)
    ref_keys = ref_group["inchikey14"].to_numpy()

    for key in np.unique(ref_keys):

        positions = np.flatnonzero(ref_keys == key)

        # Best reference match for each query spectrum
        best_scores = similarity[:, positions].max(axis=1)

        score_sums[key] += float(best_scores.sum())
        matched_query_counts[key] += len(best_scores)


ranking["spectral_score"] = ranking["inchikey14"].map(
    lambda key: (
        score_sums[key] / query_count
        if matched_query_counts[key] > 0
        else np.nan
    )
)

ranking["matched_query_spectra"] = (
    ranking["inchikey14"].map(matched_query_counts)
)

ranking = ranking.sort_values(
    ["spectral_score", "mass_error_ppm"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

# Evaluate the known structure AFTER ranking
known_positions = np.flatnonzero(
    ranking["inchikey14"].to_numpy() == true_key
)

known_rank = (
    int(known_positions[0]) + 1
    if len(known_positions) == 1
    else None
)

print("CASMI 2026 - Spectral Ranking Experiment")
print("----------------------------------------")
print("Test molecule:", molecule_id)
print("Total candidates:", len(ranking))
print(
    "Candidates with spectral scores:",
    int(ranking["spectral_score"].notna().sum())
)
print("Known structure rank:", known_rank)
print(
    "Known structure in top 25:",
    known_rank is not None and known_rank <= 25
)

print("\nTop 5 candidates:")
print(
    ranking[
        [
            "inchikey14",
            "spectral_score",
            "matched_query_spectra",
            "mass_error_ppm"
        ]
    ].head(5).to_string(index=False)
)

if known_rank is not None:
    print("\nKnown structure:")
    print(
        ranking.loc[
            known_rank - 1,
            [
                "inchikey14",
                "spectral_score",
                "matched_query_spectra",
                "mass_error_ppm"
            ]
        ].to_string()
    )

CASMI 2026 - Spectral Ranking Experiment
----------------------------------------
Test molecule: m_006153
Total candidates: 257
Candidates with spectral scores: 250
Known structure rank: 15
Known structure in top 25: True

Top 5 candidates:
    inchikey14  spectral_score  matched_query_spectra  mass_error_ppm
OWLWDUZRCJYSHF        0.443144                      3        1.203321
JFPHIOHAVGPTKT        0.371177                      6       15.639837
SSOWPHRKXKLBRF        0.357845                      6        1.203321
VYMUBUVCGNOAMN        0.328112                      6       11.719535
MDQHIPKMJWCDLW        0.327811                      6        8.369637

Known structure:
inchikey14               BKIHGKNEJKYYAC
spectral_score                 0.246311
matched_query_spectra                 3
mass_error_ppm                 1.203321


In [14]:
# CASMI 2026 - Inspect the known candidate's spectral scores

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

known_refs = experiment_references[
    experiment_references["inchikey14"] == true_key
]

print("CASMI 2026 - Known Structure Spectral Scores")
print("---------------------------------------------")
print("Test molecule:", molecule_id)
print("Known structure:", true_key)
print("Reference spectra:", len(known_refs))

for _, query_row in queries.iterrows():

    same_adduct_refs = known_refs[
        known_refs["adduct"] == query_row["adduct"]
    ]

    if same_adduct_refs.empty:
        print(
            query_row["spectrum_id"],
            "| Adduct:", query_row["adduct"],
            "| No matching reference"
        )
        continue

    query_vector = extract_features(query_row).reshape(1, -1)

    reference_vectors = np.stack(
        same_adduct_refs.apply(
            extract_features,
            axis=1
        ).to_numpy()
    )

    best_score = cosine_similarity(
        query_vector,
        reference_vectors
    ).max()

    print(
        query_row["spectrum_id"],
        "| Adduct:", query_row["adduct"],
        "| Best similarity:", round(float(best_score), 6)
    )

CASMI 2026 - Known Structure Spectral Scores
---------------------------------------------
Test molecule: m_006153
Known structure: BKIHGKNEJKYYAC
Reference spectra: 1
s_3c74e9fe | Adduct: [M+H]+ | Best similarity: 0.148354
s_54245904 | Adduct: [M-H]- | No matching reference
s_59a80abf | Adduct: [M-H]- | No matching reference
s_7151fa86 | Adduct: [M+H]+ | Best similarity: 0.592624
s_86472581 | Adduct: [M-H]- | No matching reference
s_cb15f0ac | Adduct: [M+H]+ | Best similarity: 0.736886


In [15]:
# CASMI 2026 - Compare three spectral scoring methods

import numpy as np
import pandas as pd

comparison = ranking[
    [
        "inchikey14",
        "mass_error_ppm",
        "spectral_score",
        "matched_query_spectra"
    ]
].copy()

# Method 1: Original score
# Missing query-adduct references contribute zero.
comparison["score_original"] = comparison["spectral_score"]

# Method 2: Average over query spectra that have references.
# This does not penalize missing reference adducts.
comparison["score_available"] = comparison.apply(
    lambda row: (
        score_sums[row["inchikey14"]]
        / matched_query_counts[row["inchikey14"]]
        if matched_query_counts[row["inchikey14"]] > 0
        else np.nan
    ),
    axis=1
)

# Method 3: Average over available spectra,
# with a moderate penalty for incomplete reference coverage.
coverage = comparison["matched_query_spectra"] / len(queries)

comparison["score_coverage_adjusted"] = (
    comparison["score_available"]
    * (0.5 + 0.5 * coverage)
)

print("CASMI 2026 - Scoring Method Comparison")
print("--------------------------------------")
print("Test molecule:", molecule_id)
print("Known structure:", true_key)

for method in [
    "score_original",
    "score_available",
    "score_coverage_adjusted"
]:
    ordered = comparison.sort_values(
        [method, "mass_error_ppm"],
        ascending=[False, True],
        na_position="last"
    ).reset_index(drop=True)

    positions = np.flatnonzero(
        ordered["inchikey14"].to_numpy() == true_key
    )

    known_rank = (
        int(positions[0]) + 1
        if len(positions) == 1
        else None
    )

    print(f"\nMethod: {method}")
    print("Known structure rank:", known_rank)

    if known_rank is not None:
        print(
            "Known structure score:",
            round(
                float(ordered.loc[known_rank - 1, method]),
                6
            )
        )

    print(
        "Top candidate:",
        ordered.loc[0, "inchikey14"]
    )

CASMI 2026 - Scoring Method Comparison
--------------------------------------
Test molecule: m_006153
Known structure: BKIHGKNEJKYYAC

Method: score_original
Known structure rank: 15
Known structure score: 0.246311
Top candidate: OWLWDUZRCJYSHF

Method: score_available
Known structure rank: 4
Known structure score: 0.492621
Top candidate: OWLWDUZRCJYSHF

Method: score_coverage_adjusted
Known structure rank: 5
Known structure score: 0.369466
Top candidate: OWLWDUZRCJYSHF


In [16]:
# CASMI 2026 - Check memory before batch spectral evaluation

import os
import psutil

memory = psutil.virtual_memory()
process = psutil.Process(os.getpid())

print("CASMI 2026 - Memory Check")
print("-------------------------")
print(
    "Total RAM:",
    round(memory.total / (1024 ** 3), 2),
    "GB"
)
print(
    "Available RAM:",
    round(memory.available / (1024 ** 3), 2),
    "GB"
)
print(
    "Notebook RAM usage:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)
print(
    "Training row groups:",
    train_file.num_row_groups
)

CASMI 2026 - Memory Check
-------------------------
Total RAM: 31.35 GB
Available RAM: 23.64 GB
Notebook RAM usage: 6.49 GB
Training row groups: 21


In [17]:
# CASMI 2026 - Prepare a reproducible evaluation subset

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

all_molecule_ids = sample_submission["molecule_id"].to_numpy()

evaluation_ids = rng.choice(
    all_molecule_ids,
    size=min(20, len(all_molecule_ids)),
    replace=False
).tolist()

# Use the same 20 ppm mass filter for every molecule
library_with_mass = full_candidate_library.dropna(
    subset=["reference_mass"]
)

reference_masses = library_with_mass["reference_mass"].to_numpy(
    dtype=float
)

evaluation_candidates = {}
all_candidate_keys = set()
all_evaluation_adducts = set()

for molecule_id in evaluation_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidates = library_with_mass.loc[
        ppm_errors <= 20
    ]

    candidate_keys = set(candidates["inchikey14"])

    evaluation_candidates[molecule_id] = candidate_keys
    all_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        test_spectra.loc[
            test_spectra["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    all_evaluation_adducts.update(molecule_adducts)


print("CASMI 2026 - Batch Evaluation Preparation")
print("-----------------------------------------")
print("Evaluation molecules:", len(evaluation_ids))

print(
    "Evaluation test spectra:",
    int(
        test_spectra["molecule_id"]
        .isin(evaluation_ids)
        .sum()
    )
)

print(
    "Unique candidate structures to search:",
    len(all_candidate_keys)
)

print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in evaluation_candidates.values())
)

print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in evaluation_candidates.values())
)

print(
    "Adducts in evaluation subset:",
    sorted(all_evaluation_adducts)
)

print("\nEvaluation molecules:")
print(evaluation_ids)

CASMI 2026 - Batch Evaluation Preparation
-----------------------------------------
Evaluation molecules: 20
Evaluation test spectra: 55
Unique candidate structures to search: 2306
Minimum candidates per molecule: 9
Maximum candidates per molecule: 347
Adducts in evaluation subset: ['[M+H]+', '[M+Na]+', '[M-H]-']

Evaluation molecules:
['m_c886af', 'm_bf6cfe', 'm_30024e', 'm_f866a0', 'm_e138be', 'm_c9a4bc', 'm_1de28b', 'm_153108', 'm_acdfec', 'm_8d19fa', 'm_7108be', 'm_145679', 'm_7b993b', 'm_dc7be7', 'm_15dab4', 'm_c3e52e', 'm_d07b86', 'm_741f2c', 'm_b8f328', 'm_8d75c7']


In [18]:
# CASMI 2026 - Small, memory-safe reference search test

import os
import psutil

process = psutil.Process(os.getpid())

memory_before = process.memory_info().rss / (1024 ** 3)

# Read only two metadata columns from the first 2,000 rows
small_batch = next(
    train_file.iter_batches(
        batch_size=2000,
        columns=["inchikey14", "adduct"]
    )
).to_pandas()

# Find candidate structures from our 20-molecule evaluation set
matches = small_batch[
    small_batch["inchikey14"].isin(all_candidate_keys)
    & small_batch["adduct"].isin(all_evaluation_adducts)
]

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Safe Reference Search Test")
print("---------------------------------------")
print("Rows inspected:", len(small_batch))
print("Matching rows:", len(matches))
print("Unique matching structures:", matches["inchikey14"].nunique())
print("Notebook RAM before:", round(memory_before, 3), "GB")
print("Notebook RAM after:", round(memory_after, 3), "GB")
print("RAM change:", round(memory_after - memory_before, 3), "GB")

CASMI 2026 - Safe Reference Search Test
---------------------------------------
Rows inspected: 2000
Matching rows: 2
Unique matching structures: 2
Notebook RAM before: 6.494 GB
Notebook RAM after: 6.495 GB
RAM change: 0.0 GB


In [19]:
# CASMI 2026 - Bounded reference collection test

import os
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

MAX_ROWS = 50000
MAX_REFERENCES = 500
BATCH_SIZE = 5000

memory_before = process.memory_info().rss / (1024 ** 3)

pilot_parts = []
rows_inspected = 0
references_collected = 0

columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Read a maximum of 10 batches from row group 0
for batch in train_file.iter_batches(
    row_groups=[0],
    batch_size=BATCH_SIZE,
    columns=columns
):
    batch_df = batch.to_pandas()

    rows_inspected += len(batch_df)

    matches = batch_df[
        batch_df["inchikey14"].isin(all_candidate_keys)
        & batch_df["adduct"].isin(all_evaluation_adducts)
    ]

    # Do not collect more than 500 reference spectra
    remaining_capacity = (
        MAX_REFERENCES - references_collected
    )

    if remaining_capacity > 0 and not matches.empty:
        selected = matches.head(remaining_capacity).copy()
        pilot_parts.append(selected)

        references_collected += len(selected)

    if (
        rows_inspected >= MAX_ROWS
        or references_collected >= MAX_REFERENCES
    ):
        break

pilot_references = (
    pd.concat(pilot_parts, ignore_index=True)
    if pilot_parts
    else pd.DataFrame(columns=columns)
)

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Bounded Reference Collection")
print("-----------------------------------------")
print("Rows inspected:", rows_inspected)
print("Reference spectra collected:", len(pilot_references))
print(
    "Unique candidate structures:",
    pilot_references["inchikey14"].nunique()
)

print(
    "RAM before:",
    round(memory_before, 3),
    "GB"
)

print(
    "RAM after:",
    round(memory_after, 3),
    "GB"
)

print(
    "RAM change:",
    round(memory_after - memory_before, 3),
    "GB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024 ** 3),
        3
    ),
    "GB"
)

# Free the temporary references after measuring memory
del pilot_references
del pilot_parts
del batch_df

print("\nBounded collection test completed!")

CASMI 2026 - Bounded Reference Collection
-----------------------------------------
Rows inspected: 50000
Reference spectra collected: 379
Unique candidate structures: 92
RAM before: 6.495 GB
RAM after: 6.727 GB
RAM change: 0.232 GB
Available RAM: 23.409 GB

Bounded collection test completed!


In [20]:
# CASMI 2026 - Bounded reference collection test

import os
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

MAX_ROWS = 50000
MAX_REFERENCES = 500
BATCH_SIZE = 5000

memory_before = process.memory_info().rss / (1024 ** 3)

pilot_parts = []
rows_inspected = 0
references_collected = 0

columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Read a maximum of 10 batches from row group 0
for batch in train_file.iter_batches(
    row_groups=[0],
    batch_size=BATCH_SIZE,
    columns=columns
):
    batch_df = batch.to_pandas()

    rows_inspected += len(batch_df)

    matches = batch_df[
        batch_df["inchikey14"].isin(all_candidate_keys)
        & batch_df["adduct"].isin(all_evaluation_adducts)
    ]

    # Do not collect more than 500 reference spectra
    remaining_capacity = (
        MAX_REFERENCES - references_collected
    )

    if remaining_capacity > 0 and not matches.empty:
        selected = matches.head(remaining_capacity).copy()
        pilot_parts.append(selected)

        references_collected += len(selected)

    if (
        rows_inspected >= MAX_ROWS
        or references_collected >= MAX_REFERENCES
    ):
        break

pilot_references = (
    pd.concat(pilot_parts, ignore_index=True)
    if pilot_parts
    else pd.DataFrame(columns=columns)
)

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Bounded Reference Collection")
print("-----------------------------------------")
print("Rows inspected:", rows_inspected)
print("Reference spectra collected:", len(pilot_references))
print(
    "Unique candidate structures:",
    pilot_references["inchikey14"].nunique()
)

print(
    "RAM before:",
    round(memory_before, 3),
    "GB"
)

print(
    "RAM after:",
    round(memory_after, 3),
    "GB"
)

print(
    "RAM change:",
    round(memory_after - memory_before, 3),
    "GB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024 ** 3),
        3
    ),
    "GB"
)

# Free the temporary references after measuring memory
del pilot_references
del pilot_parts
del batch_df

print("\nBounded collection test completed!")

CASMI 2026 - Bounded Reference Collection
-----------------------------------------
Rows inspected: 50000
Reference spectra collected: 379
Unique candidate structures: 92
RAM before: 6.727 GB
RAM after: 6.696 GB
RAM change: -0.031 GB
Available RAM: 23.43 GB

Bounded collection test completed!


In [21]:
# CASMI 2026 - Memory-limited reference collection

import os
import gc
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_REFERENCES = 40000
MAX_PROCESS_RAM_GB = 12.0
MIN_AVAILABLE_RAM_GB = 6.0

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Fingerprints of all test spectra in our evaluation subset
evaluation_queries = test_spectra[
    test_spectra["molecule_id"].isin(evaluation_ids)
]

evaluation_fingerprints = {
    spectrum_fingerprint(row)
    for row in evaluation_queries.itertuples(index=False)
}

reference_parts = []
references_collected = 0
exact_copies_excluded = 0
rows_inspected = 0

collection_complete = True
stop_reason = None

print("CASMI 2026 - Reference Collection")
print("---------------------------------")

for group_index in range(train_file.num_row_groups):

    memory = psutil.virtual_memory()
    ram_used = process.memory_info().rss / (1024 ** 3)
    ram_available = memory.available / (1024 ** 3)

    # Stop before reading another row group if memory is low
    if (
        ram_used >= MAX_PROCESS_RAM_GB
        or ram_available <= MIN_AVAILABLE_RAM_GB
    ):
        collection_complete = False
        stop_reason = "RAM safety limit reached"
        break

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(all_candidate_keys)
            & batch_df["adduct"].isin(all_evaluation_adducts)
        ].copy()

        if not matches.empty:

            # Remove exact copies of evaluation test spectra
            is_copy = [
                spectrum_fingerprint(row)
                in evaluation_fingerprints
                for row in matches.itertuples(index=False)
            ]

            exact_copies_excluded += sum(is_copy)

            remaining = matches.loc[
                ~pd.Series(is_copy, index=matches.index)
            ]

            if not remaining.empty:
                reference_parts.append(remaining)
                references_collected += len(remaining)

        del batch_df, matches

        # Check memory after each batch
        ram_used = process.memory_info().rss / (1024 ** 3)
        ram_available = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if references_collected >= MAX_REFERENCES:
            collection_complete = False
            stop_reason = "Reference count limit reached"
            break

        if (
            ram_used >= MAX_PROCESS_RAM_GB
            or ram_available <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {references_collected} | "
        f"RAM: {ram_used:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


evaluation_references = (
    pd.concat(reference_parts, ignore_index=True)
    if reference_parts
    else pd.DataFrame(columns=reference_columns)
)

del reference_parts
gc.collect()

print("\nCASMI 2026 - Collection Results")
print("--------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(evaluation_references))
print(
    "Unique reference structures:",
    evaluation_references["inchikey14"].nunique()
)
print("Exact copies excluded:", exact_copies_excluded)

print(
    "Current notebook RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if not collection_complete:
    print("Stopped early:", stop_reason)
else:
    print("All training row groups checked successfully!")

CASMI 2026 - Reference Collection
---------------------------------
Row groups checked: 1/21 | References: 997 | RAM: 6.99 GB
Row groups checked: 2/21 | References: 1920 | RAM: 7.44 GB
Row groups checked: 3/21 | References: 2791 | RAM: 7.94 GB
Row groups checked: 4/21 | References: 3767 | RAM: 8.41 GB
Row groups checked: 5/21 | References: 4713 | RAM: 8.88 GB
Row groups checked: 6/21 | References: 5468 | RAM: 9.41 GB
Row groups checked: 7/21 | References: 6380 | RAM: 9.88 GB
Row groups checked: 8/21 | References: 7281 | RAM: 10.39 GB
Row groups checked: 9/21 | References: 7987 | RAM: 10.85 GB
Row groups checked: 10/21 | References: 8282 | RAM: 11.40 GB
Row groups checked: 11/21 | References: 8531 | RAM: 11.50 GB
Row groups checked: 12/21 | References: 8742 | RAM: 11.91 GB
Row groups checked: 13/21 | References: 9234 | RAM: 11.95 GB
Row groups checked: 14/21 | References: 10056 | RAM: 11.79 GB
Row groups checked: 15/21 | References: 10847 | RAM: 11.87 GB
Row groups checked: 16/21 | Refe

In [22]:
# CASMI 2026 - Convert collected references to compact feature vectors

import gc
import os
import numpy as np
import psutil

process = psutil.Process(os.getpid())

ram_before = process.memory_info().rss / (1024 ** 3)

# Keep only the metadata needed for candidate ranking
reference_metadata = evaluation_references[
    ["inchikey14", "adduct"]
].copy().reset_index(drop=True)

reference_count = len(evaluation_references)

# Allocate a compact float32 matrix
reference_features = np.zeros(
    (reference_count, 1000),
    dtype=np.float32
)

# Convert one reference spectrum at a time
for i, row in enumerate(
    evaluation_references.itertuples(index=False)
):

    mz = np.asarray(row.ms2_mzs, dtype=float)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=float
    )

    valid = (
        np.isfinite(mz)
        & np.isfinite(intensity)
        & (mz >= 0)
        & (mz < 1000)
    )

    bins = mz[valid].astype(np.int32)

    # Retain the maximum intensity in each 1 Da bin
    np.maximum.at(
        reference_features[i],
        bins,
        intensity[valid].astype(np.float32)
    )

# Release the large DataFrame containing raw fragment arrays
del evaluation_references
gc.collect()

ram_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Compact Reference Features")
print("---------------------------------------")
print("Reference spectra converted:", reference_count)
print("Feature matrix shape:", reference_features.shape)

print(
    "Feature matrix size:",
    round(reference_features.nbytes / (1024 ** 2), 2),
    "MB"
)

print(
    "Metadata rows:",
    len(reference_metadata)
)

print("RAM before:", round(ram_before, 2), "GB")
print("RAM after:", round(ram_after, 2), "GB")

print("\nCompact reference features prepared!")

CASMI 2026 - Compact Reference Features
---------------------------------------
Reference spectra converted: 11798
Feature matrix shape: (11798, 1000)
Feature matrix size: 45.01 MB
Metadata rows: 11798
RAM before: 12.0 GB
RAM after: 12.05 GB

Compact reference features prepared!


In [1]:
# CASMI 2026 - Check notebook state before continuing

import os
import psutil

required_variables = [
    "train_file",
    "test_spectra",
    "evaluation_ids",
    "evaluation_candidates",
    "all_candidate_keys",
    "all_evaluation_adducts",
    "reference_features",
    "reference_metadata"
]

print("CASMI 2026 - Notebook State Check")
print("---------------------------------")

for variable_name in required_variables:
    exists = variable_name in globals()
    print(f"{variable_name}: {'Available' if exists else 'Missing'}")

process = psutil.Process(os.getpid())

print(
    "\nCurrent notebook RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

print(
    "Available Kaggle RAM:",
    round(psutil.virtual_memory().available / (1024 ** 3), 2),
    "GB"
)

CASMI 2026 - Notebook State Check
---------------------------------
train_file: Missing
test_spectra: Missing
evaluation_ids: Missing
evaluation_candidates: Missing
all_candidate_keys: Missing
all_evaluation_adducts: Missing
reference_features: Missing
reference_metadata: Missing

Current notebook RAM: 0.1 GB
Available Kaggle RAM: 30.12 GB


In [2]:
# CASMI 2026 - Restore essential notebook variables

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

dataset_folder = Path(
    "/kaggle/input/competitions/"
    "enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = dataset_folder / "train.parquet"
test_path = dataset_folder / "test.parquet"
submission_path = dataset_folder / "sample_submission.csv"

train_file = pq.ParquetFile(train_path)
test_file = pq.ParquetFile(test_path)

sample_submission = pd.read_csv(submission_path)

test_spectra = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "ms2_mzs",
        "ms2_normalized_intensities"
    ]
).to_pandas()

print("CASMI 2026 - Essential Variables Restored")
print("-----------------------------------------")
print("Training row groups:", train_file.num_row_groups)
print("Test spectra:", len(test_spectra))
print("Test molecules:", test_spectra["molecule_id"].nunique())
print("Submission rows:", len(sample_submission))
print("\nRestoration successful!")

CASMI 2026 - Essential Variables Restored
-----------------------------------------
Training row groups: 21
Test spectra: 1213
Test molecules: 400
Submission rows: 400

Restoration successful!


In [3]:
# CASMI 2026 - Restore the same 20 evaluation molecules

import numpy as np

rng = np.random.default_rng(42)

evaluation_ids = rng.choice(
    sample_submission["molecule_id"].to_numpy(),
    size=min(20, len(sample_submission)),
    replace=False
).tolist()

# Verify that we restored the original evaluation subset
expected_ids = [
    "m_c886af", "m_bf6cfe", "m_30024e", "m_f866a0",
    "m_e138be", "m_c9a4bc", "m_1de28b", "m_153108",
    "m_acdfec", "m_8d19fa", "m_7108be", "m_145679",
    "m_7b993b", "m_dc7be7", "m_15dab4", "m_c3e52e",
    "m_d07b86", "m_741f2c", "m_b8f328", "m_8d75c7"
]

assert evaluation_ids == expected_ids, (
    "The evaluation subset differs from the previous experiment."
)

evaluation_queries = test_spectra[
    test_spectra["molecule_id"].isin(evaluation_ids)
].copy()

print("CASMI 2026 - Evaluation Subset Restored")
print("---------------------------------------")
print("Evaluation molecules:", len(evaluation_ids))
print("Evaluation test spectra:", len(evaluation_queries))
print("Original evaluation subset restored: True")

CASMI 2026 - Evaluation Subset Restored
---------------------------------------
Evaluation molecules: 20
Evaluation test spectra: 55
Original evaluation subset restored: True


In [4]:
# CASMI 2026 - Restore the candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

print("CASMI 2026 - Restoring Candidate Library")
print("----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=20000,
        columns=library_columns
    ):
        batch_df = batch.to_pandas()

        batch_df = batch_df.dropna(
            subset=library_columns
        ).drop_duplicates(subset="inchikey14")

        new_rows = batch_df[
            ~batch_df["inchikey14"].isin(seen_keys)
        ]

        if not new_rows.empty:
            library_parts.append(new_rows)
            seen_keys.update(new_rows["inchikey14"])

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

del library_parts

print("\nCASMI 2026 - Candidate Library Restored")
print("---------------------------------------")
print("Unique structures:", len(full_candidate_library))
print(
    "Duplicate structures:",
    int(full_candidate_library["inchikey14"].duplicated().sum())
)

print("\nRestoration completed!")

CASMI 2026 - Restoring Candidate Library
----------------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

CASMI 2026 - Candidate Library Restored
---------------------------------------
Unique structures: 275810
Duplicate structures: 0

Restoration completed!


In [5]:
# CASMI 2026 - Restore candidate reference masses

import re
import numpy as np

atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}

def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )

full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

print("CASMI 2026 - Reference Masses Restored")
print("--------------------------------------")
print("Total candidates:", len(full_candidate_library))
print(
    "Candidates with reference mass:",
    int(full_candidate_library["reference_mass"].notna().sum())
)
print(
    "Candidates with missing reference mass:",
    int(full_candidate_library["reference_mass"].isna().sum())
)

CASMI 2026 - Reference Masses Restored
--------------------------------------
Total candidates: 275810
Candidates with reference mass: 275763
Candidates with missing reference mass: 47


In [6]:
# CASMI 2026 - Restore test neutral masses

import pandas as pd

adduct_shifts = {
    "[M+H]+": 1.007276466621,
    "[M-H]-": -1.007276466621,
    "[M+CH2O2-H]-": 44.99820284,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969401
}

mass_data = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

mass_data["adduct_shift"] = (
    mass_data["adduct"].map(adduct_shifts)
)

mass_data["neutral_mass"] = (
    pd.to_numeric(
        mass_data["precursor_mz"],
        errors="coerce"
    ) - mass_data["adduct_shift"]
)

test_molecules = (
    mass_data.groupby("molecule_id", as_index=False)
    .agg(
        median_neutral_mass=("neutral_mass", "median"),
        num_spectra=("spectrum_id", "count")
    )
)

mass_lookup = test_molecules.set_index(
    "molecule_id"
)["median_neutral_mass"]

print("CASMI 2026 - Neutral Masses Restored")
print("------------------------------------")
print("Test molecules:", len(test_molecules))
print(
    "Missing neutral masses:",
    int(test_molecules["median_neutral_mass"].isna().sum())
)
print(
    "Evaluation molecules with valid masses:",
    int(mass_lookup.loc[evaluation_ids].notna().sum())
)

CASMI 2026 - Neutral Masses Restored
------------------------------------
Test molecules: 400
Missing neutral masses: 0
Evaluation molecules with valid masses: 20


In [7]:
# CASMI 2026 - Restore candidates for the 20-molecule experiment

import numpy as np

PPM_TOLERANCE = 20

library_with_mass = full_candidate_library.dropna(
    subset=["reference_mass"]
)

reference_masses = library_with_mass[
    "reference_mass"
].to_numpy(dtype=float)

evaluation_candidates = {}
all_candidate_keys = set()
all_evaluation_adducts = set()

for molecule_id in evaluation_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidate_keys = set(
        library_with_mass.loc[
            ppm_errors <= PPM_TOLERANCE,
            "inchikey14"
        ]
    )

    evaluation_candidates[molecule_id] = candidate_keys
    all_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        test_spectra.loc[
            test_spectra["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    all_evaluation_adducts.update(molecule_adducts)

print("CASMI 2026 - Evaluation Candidates Restored")
print("-------------------------------------------")
print("Evaluation molecules:", len(evaluation_candidates))
print("Unique candidate structures:", len(all_candidate_keys))
print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in evaluation_candidates.values())
)
print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in evaluation_candidates.values())
)
print("Adducts:", sorted(all_evaluation_adducts))

assert len(evaluation_candidates) == 20
assert len(all_candidate_keys) == 2306

print("\nOriginal candidate set restored successfully!")

CASMI 2026 - Evaluation Candidates Restored
-------------------------------------------
Evaluation molecules: 20
Unique candidate structures: 2306
Minimum candidates per molecule: 9
Maximum candidates per molecule: 347
Adducts: ['[M+H]+', '[M+Na]+', '[M-H]-']

Original candidate set restored successfully!


In [8]:
# CASMI 2026 - Stream, compress and save evaluation references

import os
import gc
import hashlib
import numpy as np
import pandas as pd
import psutil

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_PROCESS_RAM_GB = 10.0
MIN_AVAILABLE_RAM_GB = 6.0
MAX_REFERENCES = 25000

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]


def spectrum_fingerprint(row):
    mz = np.asarray(row.ms2_mzs, dtype=np.float64)
    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=np.float64
    )

    digest = hashlib.blake2b(
        mz.tobytes() + intensity.tobytes(),
        digest_size=16
    ).digest()

    return (row.adduct, len(mz), digest)


def make_feature_vector(row):
    features = np.zeros(1000, dtype=np.float32)

    mz = np.asarray(row.ms2_mzs, dtype=float)
    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=float
    )

    valid = (
        np.isfinite(mz)
        & np.isfinite(intensity)
        & (mz >= 0)
        & (mz < 1000)
    )

    bins = mz[valid].astype(np.int32)

    np.maximum.at(
        features,
        bins,
        intensity[valid].astype(np.float32)
    )

    return features


# Exact copies must not enter our evaluation references
evaluation_fingerprints = {
    spectrum_fingerprint(row)
    for row in evaluation_queries.itertuples(index=False)
}

feature_parts = []
reference_keys = []
reference_adducts = []

excluded_copies = 0
rows_inspected = 0

collection_complete = True
stop_reason = None

print("CASMI 2026 - Compact Reference Collection")
print("-----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):

        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(all_candidate_keys)
            & batch_df["adduct"].isin(all_evaluation_adducts)
        ]

        for row in matches.itertuples(index=False):

            if spectrum_fingerprint(row) in evaluation_fingerprints:
                excluded_copies += 1
                continue

            if len(feature_parts) >= MAX_REFERENCES:
                collection_complete = False
                stop_reason = "Reference count limit reached"
                break

            feature_parts.append(make_feature_vector(row))
            reference_keys.append(row.inchikey14)
            reference_adducts.append(row.adduct)

        del matches, batch_df, batch

        # Check RAM after each batch
        used_ram = process.memory_info().rss / (1024 ** 3)
        available_ram = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if (
            used_ram >= MAX_PROCESS_RAM_GB
            or available_ram <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"
            break

        if not collection_complete:
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {len(feature_parts)} | "
        f"RAM: {used_ram:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


print("\nCASMI 2026 - Collection Results")
print("--------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(feature_parts))
print("Exact copies excluded:", excluded_copies)
print("Current RAM:", round(
    process.memory_info().rss / (1024 ** 3), 2
), "GB")

if collection_complete:

    reference_features = np.stack(feature_parts)

    reference_metadata = pd.DataFrame({
        "inchikey14": reference_keys,
        "adduct": reference_adducts
    })

    output_file = (
        "/kaggle/working/"
        "casmi_evaluation_references.npz"
    )

    # Save features and metadata together for future sessions
    np.savez_compressed(
        output_file,
        features=reference_features,
        inchikey14=np.asarray(reference_keys),
        adduct=np.asarray(reference_adducts),
        evaluation_ids=np.asarray(evaluation_ids)
    )

    print(
        "Feature matrix shape:",
        reference_features.shape
    )
    print(
        "Feature matrix size:",
        round(reference_features.nbytes / (1024 ** 2), 2),
        "MB"
    )
    print("Saved file:", output_file)

else:
    print("Stopped early:", stop_reason)
    print("Incomplete results were NOT saved.")

del feature_parts
gc.collect()

CASMI 2026 - Compact Reference Collection
-----------------------------------------
Row groups checked: 1/21 | References: 997 | RAM: 0.68 GB
Row groups checked: 2/21 | References: 1920 | RAM: 0.69 GB
Row groups checked: 3/21 | References: 2791 | RAM: 0.69 GB
Row groups checked: 4/21 | References: 3767 | RAM: 0.69 GB
Row groups checked: 5/21 | References: 4713 | RAM: 0.72 GB
Row groups checked: 6/21 | References: 5468 | RAM: 0.73 GB
Row groups checked: 7/21 | References: 6380 | RAM: 0.73 GB
Row groups checked: 8/21 | References: 7281 | RAM: 0.74 GB
Row groups checked: 9/21 | References: 7987 | RAM: 0.75 GB
Row groups checked: 10/21 | References: 8282 | RAM: 0.79 GB
Row groups checked: 11/21 | References: 8531 | RAM: 0.79 GB
Row groups checked: 12/21 | References: 8742 | RAM: 0.84 GB
Row groups checked: 13/21 | References: 9234 | RAM: 0.85 GB
Row groups checked: 14/21 | References: 10056 | RAM: 0.85 GB
Row groups checked: 15/21 | References: 10847 | RAM: 0.85 GB
Row groups checked: 16/2

0

In [9]:
# CASMI 2026 - Restore evaluation labels for the same 20 molecules

from collections import defaultdict
import pandas as pd

# Index only the 55 evaluation test spectra
evaluation_lookup = defaultdict(set)

for row in evaluation_queries.itertuples(index=False):
    fingerprint = spectrum_fingerprint(row)
    evaluation_lookup[fingerprint].add(row.molecule_id)

# Store structures associated with exact training-spectrum matches
evaluation_labels = defaultdict(set)

columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

print("CASMI 2026 - Restoring Evaluation Labels")
print("---------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=5000,
        columns=columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            fingerprint = spectrum_fingerprint(row)

            for molecule_id in evaluation_lookup.get(
                fingerprint, ()
            ):
                evaluation_labels[molecule_id].add(
                    (row.inchikey14, row.normalized_smiles)
                )

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Count molecules with unambiguous evaluation labels
unique_labels = {
    molecule_id: next(iter(structures))
    for molecule_id, structures in evaluation_labels.items()
    if len(structures) == 1
}

ambiguous_count = sum(
    len(structures) > 1
    for structures in evaluation_labels.values()
)

print("\nEvaluation molecules:", len(evaluation_ids))
print("Molecules with one matched structure:", len(unique_labels))
print("Molecules with multiple matched structures:", ambiguous_count)
print(
    "Molecules without a matched structure:",
    len(evaluation_ids) - len(evaluation_labels)
)

# Save labels only if all 20 molecules have one matched structure
if len(unique_labels) == len(evaluation_ids):

    evaluation_label_df = pd.DataFrame([
        {
            "molecule_id": molecule_id,
            "inchikey14": unique_labels[molecule_id][0],
            "normalized_smiles": unique_labels[molecule_id][1]
        }
        for molecule_id in evaluation_ids
    ])

    labels_path = "/kaggle/working/casmi_evaluation_labels.csv"

    evaluation_label_df.to_csv(
        labels_path,
        index=False
    )

    print("\nEvaluation labels saved:", labels_path)

else:
    print("\nSome labels are missing or ambiguous; check before evaluation.")

CASMI 2026 - Restoring Evaluation Labels
---------------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

Evaluation molecules: 20
Molecules with one matched structure: 20
Molecules with multiple matched structures: 0
Molecules without a matched structure: 0

Evaluation labels saved: /kaggle/working/casmi_evaluation_labels.csv


In [10]:
# CASMI 2026 - Evaluate four ranking methods on 20 molecules

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Labels are used ONLY to evaluate completed rankings
labels = pd.read_csv(
    "/kaggle/working/casmi_evaluation_labels.csv"
).set_index("molecule_id")["inchikey14"]

ref_keys = reference_metadata["inchikey14"].to_numpy()
ref_adducts = reference_metadata["adduct"].to_numpy()

methods = [
    "mass_only",
    "original",
    "available",
    "coverage_adjusted"
]

results = []

print("CASMI 2026 - Evaluating 20 Molecules")
print("------------------------------------")

for molecule_id in evaluation_ids:

    query_group = evaluation_queries[
        evaluation_queries["molecule_id"] == molecule_id
    ]

    query_mass = float(mass_lookup.loc[molecule_id])
    candidate_keys = evaluation_candidates[molecule_id]
    known_key = labels.loc[molecule_id]

    # Candidate metadata and mass errors
    candidates = library_with_mass[
        library_with_mass["inchikey14"].isin(candidate_keys)
    ][["inchikey14", "reference_mass"]].copy()

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    score_sums = {key: 0.0 for key in candidate_keys}
    score_counts = {key: 0 for key in candidate_keys}

    # Select references for this molecule only
    candidate_mask = np.isin(
        ref_keys, list(candidate_keys)
    )

    for adduct, adduct_queries in query_group.groupby("adduct"):

        ref_positions = np.flatnonzero(
            candidate_mask & (ref_adducts == adduct)
        )

        if len(ref_positions) == 0:
            continue

        # Small matrices: only this molecule and this adduct
        X_query = np.stack([
            make_feature_vector(row)
            for row in adduct_queries.itertuples(index=False)
        ])

        X_ref = reference_features[ref_positions]

        similarities = cosine_similarity(X_query, X_ref)
        selected_keys = ref_keys[ref_positions]

        for key in np.unique(selected_keys):

            key_positions = np.flatnonzero(
                selected_keys == key
            )

            best_per_query = similarities[
                :, key_positions
            ].max(axis=1)

            score_sums[key] += float(best_per_query.sum())
            score_counts[key] += len(best_per_query)

    total_queries = len(query_group)

    candidates["matched_queries"] = candidates[
        "inchikey14"
    ].map(score_counts)

    candidates["original"] = candidates["inchikey14"].map(
        lambda key: (
            score_sums[key] / total_queries
            if score_counts[key] > 0
            else np.nan
        )
    )

    candidates["available"] = candidates["inchikey14"].map(
        lambda key: (
            score_sums[key] / score_counts[key]
            if score_counts[key] > 0
            else np.nan
        )
    )

    coverage = candidates["matched_queries"] / total_queries

    candidates["coverage_adjusted"] = (
        candidates["available"] * (0.5 + 0.5 * coverage)
    )

    # Evaluate each method AFTER ranking
    for method in methods:

        if method == "mass_only":
            ordered = candidates.sort_values(
                ["mass_error_ppm", "inchikey14"],
                ascending=[True, True]
            )
        else:
            ordered = candidates.sort_values(
                [method, "mass_error_ppm", "inchikey14"],
                ascending=[False, True, True],
                na_position="last"
            )

        ranked_keys = ordered["inchikey14"].to_numpy()

        positions = np.flatnonzero(ranked_keys == known_key)

        known_rank = (
            int(positions[0]) + 1
            if len(positions) == 1
            else None
        )

        results.append({
            "molecule_id": molecule_id,
            "method": method,
            "known_rank": known_rank,
            "reciprocal_rank_at_25": (
                1.0 / known_rank
                if known_rank is not None and known_rank <= 25
                else 0.0
            ),
            "top1": known_rank == 1,
            "top25": (
                known_rank is not None and known_rank <= 25
            )
        })

    print(f"Evaluated: {molecule_id}")

evaluation_results_df = pd.DataFrame(results)

summary = (
    evaluation_results_df.groupby("method", sort=False)
    .agg(
        molecules=("molecule_id", "count"),
        top1_matches=("top1", "sum"),
        top25_matches=("top25", "sum"),
        mean_reciprocal_rank_at_25=(
            "reciprocal_rank_at_25", "mean"
        )
    )
)

print("\nCASMI 2026 - Evaluation Results")
print("--------------------------------")
print(summary.to_string())

print("\nResults for m_006153:")
print(
    evaluation_results_df.loc[
        evaluation_results_df["molecule_id"] == "m_006153",
        ["method", "known_rank"]
    ].to_string(index=False)
)

CASMI 2026 - Evaluating 20 Molecules
------------------------------------
Evaluated: m_c886af
Evaluated: m_bf6cfe
Evaluated: m_30024e
Evaluated: m_f866a0
Evaluated: m_e138be
Evaluated: m_c9a4bc
Evaluated: m_1de28b
Evaluated: m_153108
Evaluated: m_acdfec
Evaluated: m_8d19fa
Evaluated: m_7108be
Evaluated: m_145679
Evaluated: m_7b993b
Evaluated: m_dc7be7
Evaluated: m_15dab4
Evaluated: m_c3e52e
Evaluated: m_d07b86
Evaluated: m_741f2c
Evaluated: m_b8f328
Evaluated: m_8d75c7

CASMI 2026 - Evaluation Results
--------------------------------
                   molecules  top1_matches  top25_matches  mean_reciprocal_rank_at_25
method                                                                               
mass_only                 20             1             13                    0.141274
original                  20             7             10                    0.416667
available                 20             5             10                    0.358333
coverage_adjusted         20  

In [11]:
# CASMI 2026 - Compare rankings molecule by molecule

import pandas as pd

rank_comparison = evaluation_results_df.pivot(
    index="molecule_id",
    columns="method",
    values="known_rank"
).reset_index()

rank_comparison["rank_change"] = (
    rank_comparison["mass_only"]
    - rank_comparison["original"]
)

rank_comparison["mass_top25"] = (
    rank_comparison["mass_only"] <= 25
)

rank_comparison["spectral_top25"] = (
    rank_comparison["original"] <= 25
)

print("CASMI 2026 - Per-Molecule Ranking Comparison")
print("---------------------------------------------")

print(
    rank_comparison[
        [
            "molecule_id",
            "mass_only",
            "original",
            "available",
            "coverage_adjusted",
            "rank_change"
        ]
    ].to_string(index=False)
)

print("\nTop-25 comparison:")

print(
    "Improved from outside to inside top 25:",
    int((
        ~rank_comparison["mass_top25"]
        & rank_comparison["spectral_top25"]
    ).sum())
)

print(
    "Dropped from inside to outside top 25:",
    int((
        rank_comparison["mass_top25"]
        & ~rank_comparison["spectral_top25"]
    ).sum())
)

print(
    "Inside top 25 with both methods:",
    int((
        rank_comparison["mass_top25"]
        & rank_comparison["spectral_top25"]
    ).sum())
)

CASMI 2026 - Per-Molecule Ranking Comparison
---------------------------------------------
molecule_id  mass_only  original  available  coverage_adjusted  rank_change
   m_145679         75         2          2                  2           73
   m_153108          2         1          1                  1            1
   m_15dab4         23        49         49                 49          -26
   m_1de28b         28         2          2                  2           26
   m_30024e         26        64         64                 64          -38
   m_7108be          8        28         28                 28          -20
   m_741f2c         22        57         57                 57          -35
   m_7b993b        125       223        223                223          -98
   m_8d19fa         19        39         39                 39          -20
   m_8d75c7          1         1          1                  1            0
   m_acdfec        254         1          1                  1          2

In [12]:
# CASMI 2026 - Diagnose candidates lost by spectral ranking

dropped = rank_comparison[
    rank_comparison["mass_top25"]
    & ~rank_comparison["spectral_top25"]
].copy()

print("CASMI 2026 - Lost Top-25 Candidates")
print("-----------------------------------")

for row in dropped.itertuples(index=False):

    molecule_id = row.molecule_id

    known_key = labels.loc[molecule_id]

    query_adducts = set(
        evaluation_queries.loc[
            evaluation_queries["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    known_refs = reference_metadata[
        reference_metadata["inchikey14"] == known_key
    ]

    matching_refs = known_refs[
        known_refs["adduct"].isin(query_adducts)
    ]

    print("\nMolecule:", molecule_id)
    print("Mass-only rank:", row.mass_only)
    print("Spectral rank:", row.original)
    print("Known structure:", known_key)
    print("Test adducts:", sorted(query_adducts))
    print("Reference spectra for known structure:", len(known_refs))
    print("References with matching adduct:", len(matching_refs))

    if not matching_refs.empty:
        print(
            "Matching reference adduct counts:",
            matching_refs["adduct"].value_counts().to_dict()
        )
    else:
        print("No matching-adduct references available.")

print("\nTotal molecules inspected:", len(dropped))

CASMI 2026 - Lost Top-25 Candidates
-----------------------------------

Molecule: m_15dab4
Mass-only rank: 23
Spectral rank: 49
Known structure: ZCHLWNAFOZCNOT
Test adducts: ['[M+H]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Molecule: m_7108be
Mass-only rank: 8
Spectral rank: 28
Known structure: HMBVMJFEDBMGPC
Test adducts: ['[M+H]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Molecule: m_741f2c
Mass-only rank: 22
Spectral rank: 57
Known structure: OXTXLZOHBPTDMJ
Test adducts: ['[M+H]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Molecule: m_8d19fa
Mass-only rank: 19
Spectral rank: 39
Known structure: BRJPAXQZDQTQRY
Test adducts: ['[M+Na]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Mol

In [13]:
# CASMI 2026 - Evaluate hybrid mass and spectral ranking

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

hybrid_results = []

ref_keys = reference_metadata["inchikey14"].to_numpy()
ref_adducts = reference_metadata["adduct"].to_numpy()

print("CASMI 2026 - Hybrid Ranking Evaluation")
print("--------------------------------------")

for molecule_id in evaluation_ids:

    queries = evaluation_queries[
        evaluation_queries["molecule_id"] == molecule_id
    ]

    query_mass = float(mass_lookup.loc[molecule_id])
    candidate_keys = evaluation_candidates[molecule_id]

    candidates = library_with_mass[
        library_with_mass["inchikey14"].isin(candidate_keys)
    ][["inchikey14", "reference_mass"]].copy()

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    score_sums = {key: 0.0 for key in candidate_keys}
    score_counts = {key: 0 for key in candidate_keys}

    candidate_mask = np.isin(
        ref_keys, list(candidate_keys)
    )

    # Compare spectra with the same adduct
    for adduct, query_group in queries.groupby("adduct"):

        positions = np.flatnonzero(
            candidate_mask & (ref_adducts == adduct)
        )

        if len(positions) == 0:
            continue

        X_query = np.stack([
            make_feature_vector(row)
            for row in query_group.itertuples(index=False)
        ])

        X_ref = reference_features[positions]

        similarities = cosine_similarity(X_query, X_ref)
        selected_keys = ref_keys[positions]

        for key in np.unique(selected_keys):

            key_positions = np.flatnonzero(
                selected_keys == key
            )

            best_scores = similarities[
                :, key_positions
            ].max(axis=1)

            score_sums[key] += float(best_scores.sum())
            score_counts[key] += len(best_scores)

    # Original spectral scores
    candidates["spectral_score"] = candidates[
        "inchikey14"
    ].map(
        lambda key: (
            score_sums[key] / len(queries)
            if score_counts[key] > 0
            else np.nan
        )
    )

    # Rank every candidate by mass
    mass_order = candidates.sort_values(
        ["mass_error_ppm", "inchikey14"]
    ).reset_index(drop=True)

    mass_ranks = {
        key: rank
        for rank, key in enumerate(
            mass_order["inchikey14"], start=1
        )
    }

    # Rank only candidates with spectral evidence
    spectral_order = candidates.dropna(
        subset=["spectral_score"]
    ).sort_values(
        ["spectral_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True]
    )

    spectral_ranks = {
        key: rank
        for rank, key in enumerate(
            spectral_order["inchikey14"], start=1
        )
    }

    # No spectral reference: retain the mass rank
    # instead of treating missing evidence as poor similarity.
    def hybrid_score(key):
        mass_rank = mass_ranks[key]
        spectral_rank = spectral_ranks.get(key, mass_rank)

        return (
            0.7 / (10 + mass_rank)
            + 0.3 / (10 + spectral_rank)
        )

    candidates["hybrid_score"] = candidates[
        "inchikey14"
    ].map(hybrid_score)

    hybrid_order = candidates.sort_values(
        ["hybrid_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    # Use the known structure ONLY after ranking
    known_key = labels.loc[molecule_id]

    positions = np.flatnonzero(
        hybrid_order["inchikey14"].to_numpy() == known_key
    )

    known_rank = (
        int(positions[0]) + 1
        if len(positions) == 1
        else None
    )

    hybrid_results.append({
        "molecule_id": molecule_id,
        "known_rank": known_rank,
        "top1": known_rank == 1,
        "top25": known_rank is not None and known_rank <= 25,
        "reciprocal_rank_at_25": (
            1.0 / known_rank
            if known_rank is not None and known_rank <= 25
            else 0.0
        )
    })

hybrid_results_df = pd.DataFrame(hybrid_results)

print("\nCASMI 2026 - Hybrid Results")
print("---------------------------")
print("Molecules evaluated:", len(hybrid_results_df))
print("Top 1:", int(hybrid_results_df["top1"].sum()))
print("Top 25:", int(hybrid_results_df["top25"].sum()))
print(
    "MRR@25:",
    round(
        hybrid_results_df["reciprocal_rank_at_25"].mean(),
        6
    )
)

print("\nRanks of the 7 previously lost molecules:")

print(
    hybrid_results_df[
        hybrid_results_df["molecule_id"].isin(
            dropped["molecule_id"]
        )
    ][["molecule_id", "known_rank"]].to_string(index=False)
)

CASMI 2026 - Hybrid Ranking Evaluation
--------------------------------------

CASMI 2026 - Hybrid Results
---------------------------
Molecules evaluated: 20
Top 1: 3
Top 25: 16
MRR@25: 0.233314

Ranks of the 7 previously lost molecules:
molecule_id  known_rank
   m_f866a0           5
   m_e138be          22
   m_8d19fa          20
   m_7108be          10
   m_dc7be7          12
   m_15dab4          30
   m_741f2c          24


In [14]:
# CASMI 2026 - Compare hybrid and mass-only rankings

import pandas as pd

# Get mass-only ranks from our previous evaluation
mass_ranks = evaluation_results_df[
    evaluation_results_df["method"] == "mass_only"
][["molecule_id", "known_rank"]].rename(
    columns={"known_rank": "mass_rank"}
)

# Get hybrid ranks
hybrid_ranks = hybrid_results_df[
    ["molecule_id", "known_rank"]
].rename(
    columns={"known_rank": "hybrid_rank"}
)

comparison = mass_ranks.merge(
    hybrid_ranks,
    on="molecule_id",
    validate="one_to_one"
)

comparison["mass_top25"] = comparison["mass_rank"] <= 25
comparison["hybrid_top25"] = comparison["hybrid_rank"] <= 25

print("CASMI 2026 - Hybrid vs Mass-Only")
print("--------------------------------")

print(
    "Improved into Top 25:",
    int((
        ~comparison["mass_top25"]
        & comparison["hybrid_top25"]
    ).sum())
)

print(
    "Dropped out of Top 25:",
    int((
        comparison["mass_top25"]
        & ~comparison["hybrid_top25"]
    ).sum())
)

print(
    "Inside Top 25 with both:",
    int((
        comparison["mass_top25"]
        & comparison["hybrid_top25"]
    ).sum())
)

print("\nMolecules improved into Top 25:")
print(
    comparison.loc[
        ~comparison["mass_top25"]
        & comparison["hybrid_top25"],
        ["molecule_id", "mass_rank", "hybrid_rank"]
    ].to_string(index=False)
)

print("\nMolecules dropped out of Top 25:")
print(
    comparison.loc[
        comparison["mass_top25"]
        & ~comparison["hybrid_top25"],
        ["molecule_id", "mass_rank", "hybrid_rank"]
    ].to_string(index=False)
)

CASMI 2026 - Hybrid vs Mass-Only
--------------------------------
Improved into Top 25: 4
Dropped out of Top 25: 1
Inside Top 25 with both: 12

Molecules improved into Top 25:
molecule_id  mass_rank  hybrid_rank
   m_bf6cfe         54           15
   m_1de28b         28           10
   m_acdfec        254           18
   m_145679         75           16

Molecules dropped out of Top 25:
molecule_id  mass_rank  hybrid_rank
   m_15dab4         23           30


In [15]:
# CASMI 2026 - Prepare an independent evaluation subset

import numpy as np

rng = np.random.default_rng(123)

# Exclude all 20 molecules used in our previous experiments
available_ids = sorted(
    set(sample_submission["molecule_id"])
    - set(evaluation_ids)
)

holdout_ids = rng.choice(
    available_ids,
    size=30,
    replace=False
).tolist()

holdout_queries = test_spectra[
    test_spectra["molecule_id"].isin(holdout_ids)
].copy()

# Verify that the two evaluation groups do not overlap
overlap = set(holdout_ids) & set(evaluation_ids)

print("CASMI 2026 - Independent Evaluation Subset")
print("------------------------------------------")
print("Previous evaluation molecules:", len(evaluation_ids))
print("New evaluation molecules:", len(holdout_ids))
print("New evaluation test spectra:", len(holdout_queries))
print("Overlapping molecules:", len(overlap))

assert len(overlap) == 0
assert len(holdout_ids) == 30

print("\nNew evaluation subset prepared successfully!")

CASMI 2026 - Independent Evaluation Subset
------------------------------------------
Previous evaluation molecules: 20
New evaluation molecules: 30
New evaluation test spectra: 90
Overlapping molecules: 0

New evaluation subset prepared successfully!


In [16]:
# CASMI 2026 - Prepare candidates for the 30-molecule holdout

import numpy as np

PPM_TOLERANCE = 20

holdout_candidates = {}
holdout_candidate_keys = set()
holdout_adducts = set()

# Reuse the candidate library and masses already in memory
reference_masses = library_with_mass[
    "reference_mass"
].to_numpy(dtype=float)

for molecule_id in holdout_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidate_keys = set(
        library_with_mass.loc[
            ppm_errors <= PPM_TOLERANCE,
            "inchikey14"
        ]
    )

    holdout_candidates[molecule_id] = candidate_keys
    holdout_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        holdout_queries.loc[
            holdout_queries["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    holdout_adducts.update(molecule_adducts)

print("CASMI 2026 - Holdout Candidates")
print("--------------------------------")
print("Holdout molecules:", len(holdout_candidates))
print("Holdout test spectra:", len(holdout_queries))
print(
    "Unique candidate structures:",
    len(holdout_candidate_keys)
)
print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in holdout_candidates.values())
)
print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in holdout_candidates.values())
)
print("Adducts:", sorted(holdout_adducts))

assert len(holdout_candidates) == 30

print("\nHoldout candidates prepared successfully!")

CASMI 2026 - Holdout Candidates
--------------------------------
Holdout molecules: 30
Holdout test spectra: 90
Unique candidate structures: 5172
Minimum candidates per molecule: 30
Maximum candidates per molecule: 381
Adducts: ['[M+H]+', '[M+NH4]+', '[M+Na]+', '[M-H]-']

Holdout candidates prepared successfully!


In [17]:
# CASMI 2026 - Collect compact references for the 30-molecule holdout

import gc
import os
import numpy as np
import pandas as pd
import psutil

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_PROCESS_RAM_GB = 6.0
MIN_AVAILABLE_RAM_GB = 6.0
MAX_REFERENCES = 60000

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Exclude exact copies of the 90 holdout test spectra
holdout_fingerprints = {
    spectrum_fingerprint(row)
    for row in holdout_queries.itertuples(index=False)
}

holdout_feature_parts = []
holdout_ref_keys = []
holdout_ref_adducts = []

excluded_copies = 0
rows_inspected = 0
collection_complete = True
stop_reason = None

print("CASMI 2026 - Holdout Reference Collection")
print("-----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(holdout_candidate_keys)
            & batch_df["adduct"].isin(holdout_adducts)
        ]

        for row in matches.itertuples(index=False):

            if spectrum_fingerprint(row) in holdout_fingerprints:
                excluded_copies += 1
                continue

            if len(holdout_feature_parts) >= MAX_REFERENCES:
                collection_complete = False
                stop_reason = "Reference count limit reached"
                break

            holdout_feature_parts.append(
                make_feature_vector(row)
            )
            holdout_ref_keys.append(row.inchikey14)
            holdout_ref_adducts.append(row.adduct)

        del matches, batch_df, batch

        used_ram = process.memory_info().rss / (1024 ** 3)
        available_ram = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if (
            used_ram >= MAX_PROCESS_RAM_GB
            or available_ram <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"

        if not collection_complete:
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {len(holdout_feature_parts)} | "
        f"RAM: {used_ram:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


print("\nCASMI 2026 - Holdout Collection Results")
print("---------------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(holdout_feature_parts))
print("Exact copies excluded:", excluded_copies)
print(
    "Current RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if collection_complete and holdout_feature_parts:

    holdout_reference_features = np.stack(
        holdout_feature_parts
    )

    holdout_reference_metadata = pd.DataFrame({
        "inchikey14": holdout_ref_keys,
        "adduct": holdout_ref_adducts
    })

    holdout_output_path = (
        "/kaggle/working/casmi_holdout_references.npz"
    )

    np.savez_compressed(
        holdout_output_path,
        features=holdout_reference_features,
        inchikey14=np.asarray(holdout_ref_keys),
        adduct=np.asarray(holdout_ref_adducts),
        holdout_ids=np.asarray(holdout_ids)
    )

    print(
        "Feature matrix shape:",
        holdout_reference_features.shape
    )
    print(
        "Feature matrix size:",
        round(
            holdout_reference_features.nbytes / (1024 ** 2),
            2
        ),
        "MB"
    )
    print("Saved file:", holdout_output_path)

else:
    print("Stopped early:", stop_reason)
    print("Incomplete reference data was not saved.")

del holdout_feature_parts
gc.collect()

CASMI 2026 - Holdout Reference Collection
-----------------------------------------
Row groups checked: 1/21 | References: 2197 | RAM: 1.10 GB
Row groups checked: 2/21 | References: 4351 | RAM: 1.10 GB
Row groups checked: 3/21 | References: 6607 | RAM: 1.10 GB
Row groups checked: 4/21 | References: 8835 | RAM: 1.11 GB
Row groups checked: 5/21 | References: 10969 | RAM: 1.11 GB
Row groups checked: 6/21 | References: 13039 | RAM: 1.11 GB
Row groups checked: 7/21 | References: 15012 | RAM: 1.12 GB
Row groups checked: 8/21 | References: 17361 | RAM: 1.13 GB
Row groups checked: 9/21 | References: 19058 | RAM: 1.14 GB
Row groups checked: 10/21 | References: 19560 | RAM: 1.14 GB
Row groups checked: 11/21 | References: 20209 | RAM: 1.14 GB
Row groups checked: 12/21 | References: 20648 | RAM: 1.15 GB
Row groups checked: 13/21 | References: 21659 | RAM: 1.15 GB
Row groups checked: 14/21 | References: 23025 | RAM: 1.15 GB
Row groups checked: 15/21 | References: 24345 | RAM: 1.16 GB
Row groups che

0

In [18]:
# CASMI 2026 - Collect compact references for the 30-molecule holdout

import gc
import os
import numpy as np
import pandas as pd
import psutil

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_PROCESS_RAM_GB = 6.0
MIN_AVAILABLE_RAM_GB = 6.0
MAX_REFERENCES = 60000

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Exclude exact copies of the 90 holdout test spectra
holdout_fingerprints = {
    spectrum_fingerprint(row)
    for row in holdout_queries.itertuples(index=False)
}

holdout_feature_parts = []
holdout_ref_keys = []
holdout_ref_adducts = []

excluded_copies = 0
rows_inspected = 0
collection_complete = True
stop_reason = None

print("CASMI 2026 - Holdout Reference Collection")
print("-----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(holdout_candidate_keys)
            & batch_df["adduct"].isin(holdout_adducts)
        ]

        for row in matches.itertuples(index=False):

            if spectrum_fingerprint(row) in holdout_fingerprints:
                excluded_copies += 1
                continue

            if len(holdout_feature_parts) >= MAX_REFERENCES:
                collection_complete = False
                stop_reason = "Reference count limit reached"
                break

            holdout_feature_parts.append(
                make_feature_vector(row)
            )
            holdout_ref_keys.append(row.inchikey14)
            holdout_ref_adducts.append(row.adduct)

        del matches, batch_df, batch

        used_ram = process.memory_info().rss / (1024 ** 3)
        available_ram = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if (
            used_ram >= MAX_PROCESS_RAM_GB
            or available_ram <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"

        if not collection_complete:
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {len(holdout_feature_parts)} | "
        f"RAM: {used_ram:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


print("\nCASMI 2026 - Holdout Collection Results")
print("---------------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(holdout_feature_parts))
print("Exact copies excluded:", excluded_copies)
print(
    "Current RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if collection_complete and holdout_feature_parts:

    holdout_reference_features = np.stack(
        holdout_feature_parts
    )

    holdout_reference_metadata = pd.DataFrame({
        "inchikey14": holdout_ref_keys,
        "adduct": holdout_ref_adducts
    })

    holdout_output_path = (
        "/kaggle/working/casmi_holdout_references.npz"
    )

    np.savez_compressed(
        holdout_output_path,
        features=holdout_reference_features,
        inchikey14=np.asarray(holdout_ref_keys),
        adduct=np.asarray(holdout_ref_adducts),
        holdout_ids=np.asarray(holdout_ids)
    )

    print(
        "Feature matrix shape:",
        holdout_reference_features.shape
    )
    print(
        "Feature matrix size:",
        round(
            holdout_reference_features.nbytes / (1024 ** 2),
            2
        ),
        "MB"
    )
    print("Saved file:", holdout_output_path)

else:
    print("Stopped early:", stop_reason)
    print("Incomplete reference data was not saved.")

del holdout_feature_parts
gc.collect()

CASMI 2026 - Holdout Reference Collection
-----------------------------------------
Row groups checked: 1/21 | References: 2197 | RAM: 1.30 GB
Row groups checked: 2/21 | References: 4351 | RAM: 1.30 GB
Row groups checked: 3/21 | References: 6607 | RAM: 1.30 GB
Row groups checked: 4/21 | References: 8835 | RAM: 1.30 GB
Row groups checked: 5/21 | References: 10969 | RAM: 1.30 GB
Row groups checked: 6/21 | References: 13039 | RAM: 1.30 GB
Row groups checked: 7/21 | References: 15012 | RAM: 1.30 GB
Row groups checked: 8/21 | References: 17361 | RAM: 1.30 GB
Row groups checked: 9/21 | References: 19058 | RAM: 1.30 GB
Row groups checked: 10/21 | References: 19560 | RAM: 1.30 GB
Row groups checked: 11/21 | References: 20209 | RAM: 1.30 GB
Row groups checked: 12/21 | References: 20648 | RAM: 1.30 GB
Row groups checked: 13/21 | References: 21659 | RAM: 1.30 GB
Row groups checked: 14/21 | References: 23025 | RAM: 1.30 GB
Row groups checked: 15/21 | References: 24345 | RAM: 1.30 GB
Row groups che

0

In [19]:
# CASMI 2026 - Restore labels for the 30-molecule holdout

from collections import defaultdict
import pandas as pd

# Link each holdout spectrum fingerprint to its molecule ID
holdout_lookup = defaultdict(set)

for row in holdout_queries.itertuples(index=False):
    holdout_lookup[spectrum_fingerprint(row)].add(
        row.molecule_id
    )

# Avoid fingerprinting training spectra with irrelevant adducts
# and peak counts.
holdout_combinations = {
    (row.adduct, len(row.ms2_mzs))
    for row in holdout_queries.itertuples(index=False)
}

holdout_label_hits = defaultdict(set)

columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

print("CASMI 2026 - Holdout Label Recovery")
print("-----------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=5000,
        columns=columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            if (
                row.adduct,
                len(row.ms2_mzs)
            ) not in holdout_combinations:
                continue

            fingerprint = spectrum_fingerprint(row)

            for molecule_id in holdout_lookup.get(
                fingerprint, ()
            ):
                holdout_label_hits[molecule_id].add(
                    (row.inchikey14, row.normalized_smiles)
                )

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Check whether each molecule has one unambiguous structure
unique_labels = {
    molecule_id: next(iter(structures))
    for molecule_id, structures in holdout_label_hits.items()
    if len(structures) == 1
}

ambiguous_count = sum(
    len(structures) > 1
    for structures in holdout_label_hits.values()
)

print("\nCASMI 2026 - Holdout Label Results")
print("----------------------------------")
print("Holdout molecules:", len(holdout_ids))
print("Molecules with one structure:", len(unique_labels))
print("Molecules with multiple structures:", ambiguous_count)
print(
    "Molecules without a match:",
    len(holdout_ids) - len(holdout_label_hits)
)

if len(unique_labels) == len(holdout_ids):

    holdout_label_df = pd.DataFrame([
        {
            "molecule_id": molecule_id,
            "inchikey14": unique_labels[molecule_id][0],
            "normalized_smiles": unique_labels[molecule_id][1]
        }
        for molecule_id in holdout_ids
    ])

    labels_path = (
        "/kaggle/working/casmi_holdout_labels.csv"
    )

    holdout_label_df.to_csv(labels_path, index=False)

    print("\nLabels saved:", labels_path)

else:
    print(
        "\nSome labels are missing or ambiguous. "
        "Review them before evaluating the model."
    )

CASMI 2026 - Holdout Label Recovery
-----------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

CASMI 2026 - Holdout Label Results
----------------------------------
Holdout molecules: 30
Molecules with one structure: 30
Molecules with multiple structures: 0
Molecules without a match: 0

Labels saved: /kaggle/working/casmi_holdout_labels.csv


In [20]:
# CASMI 2026 - Evaluate ranking methods on 30 holdout molecules

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load evaluation labels saved in our previous step
holdout_labels = pd.read_csv(
    "/kaggle/working/casmi_holdout_labels.csv"
).set_index("molecule_id")["inchikey14"]

ref_keys = holdout_reference_metadata["inchikey14"].to_numpy()
ref_adducts = holdout_reference_metadata["adduct"].to_numpy()

holdout_results = []

print("CASMI 2026 - Holdout Ranking Evaluation")
print("---------------------------------------")

for molecule_id in holdout_ids:

    queries = holdout_queries[
        holdout_queries["molecule_id"] == molecule_id
    ]

    query_mass = float(mass_lookup.loc[molecule_id])
    candidate_keys = holdout_candidates[molecule_id]

    candidates = library_with_mass[
        library_with_mass["inchikey14"].isin(candidate_keys)
    ][["inchikey14", "reference_mass"]].copy()

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    score_sums = {key: 0.0 for key in candidate_keys}
    score_counts = {key: 0 for key in candidate_keys}

    candidate_mask = np.isin(
        ref_keys,
        list(candidate_keys)
    )

    # Compare test and reference spectra with matching adducts
    for adduct, query_group in queries.groupby("adduct"):

        ref_positions = np.flatnonzero(
            candidate_mask & (ref_adducts == adduct)
        )

        if len(ref_positions) == 0:
            continue

        X_query = np.stack([
            make_feature_vector(row)
            for row in query_group.itertuples(index=False)
        ])

        X_ref = holdout_reference_features[ref_positions]

        similarities = cosine_similarity(X_query, X_ref)
        selected_keys = ref_keys[ref_positions]

        for key in np.unique(selected_keys):

            positions = np.flatnonzero(selected_keys == key)

            best_scores = similarities[
                :, positions
            ].max(axis=1)

            score_sums[key] += float(best_scores.sum())
            score_counts[key] += len(best_scores)

    # Calculate spectral scores
    candidates["spectral_score"] = candidates[
        "inchikey14"
    ].map(
        lambda key: (
            score_sums[key] / len(queries)
            if score_counts[key] > 0
            else np.nan
        )
    )

    # Mass-only ranking
    mass_order = candidates.sort_values(
        ["mass_error_ppm", "inchikey14"],
        ascending=[True, True]
    ).reset_index(drop=True)

    mass_ranks = {
        key: rank
        for rank, key in enumerate(
            mass_order["inchikey14"], start=1
        )
    }

    # Spectral-only ranking
    spectral_order = candidates.sort_values(
        ["spectral_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    # Assign spectral ranks only where reference evidence exists
    spectral_ranks = {
        key: rank
        for rank, key in enumerate(
            spectral_order.dropna(
                subset=["spectral_score"]
            )["inchikey14"],
            start=1
        )
    }

    # Keep our previously chosen hybrid formula unchanged
    def hybrid_score(key):

        mass_rank = mass_ranks[key]

        # Missing spectral evidence: retain mass rank
        spectral_rank = spectral_ranks.get(key, mass_rank)

        return (
            0.7 / (10 + mass_rank)
            + 0.3 / (10 + spectral_rank)
        )

    candidates["hybrid_score"] = candidates[
        "inchikey14"
    ].map(hybrid_score)

    hybrid_order = candidates.sort_values(
        ["hybrid_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    known_key = holdout_labels.loc[molecule_id]

    # Evaluate only after completing each ranking
    for method, ordered in [
        ("mass_only", mass_order),
        ("spectral_only", spectral_order),
        ("hybrid", hybrid_order)
    ]:

        positions = np.flatnonzero(
            ordered["inchikey14"].to_numpy() == known_key
        )

        known_rank = (
            int(positions[0]) + 1
            if len(positions) == 1
            else None
        )

        holdout_results.append({
            "molecule_id": molecule_id,
            "method": method,
            "known_rank": known_rank,
            "top1": known_rank == 1,
            "top25": (
                known_rank is not None
                and known_rank <= 25
            ),
            "reciprocal_rank_at_25": (
                1.0 / known_rank
                if known_rank is not None and known_rank <= 25
                else 0.0
            )
        })

    print("Evaluated:", molecule_id)


holdout_results_df = pd.DataFrame(holdout_results)

holdout_summary = (
    holdout_results_df.groupby("method")
    .agg(
        molecules=("molecule_id", "count"),
        top1_matches=("top1", "sum"),
        top25_matches=("top25", "sum"),
        mrr_at_25=("reciprocal_rank_at_25", "mean")
    )
)

print("\nCASMI 2026 - Holdout Evaluation Results")
print("---------------------------------------")
print(holdout_summary.to_string())

print("\nHoldout evaluation completed!")

CASMI 2026 - Holdout Ranking Evaluation
---------------------------------------
Evaluated: m_cfb3df
Evaluated: m_3db789
Evaluated: m_fea652
Evaluated: m_370436
Evaluated: m_3dc915
Evaluated: m_53dfe2
Evaluated: m_2aa8ea
Evaluated: m_ace790
Evaluated: m_e993ec
Evaluated: m_0c4e10
Evaluated: m_cd50d8
Evaluated: m_d56a4e
Evaluated: m_762dcf
Evaluated: m_d274b6
Evaluated: m_2b7513
Evaluated: m_02d188
Evaluated: m_95e8da
Evaluated: m_43e809
Evaluated: m_89a6f2
Evaluated: m_444a56
Evaluated: m_57bec0
Evaluated: m_c8934f
Evaluated: m_6f3a5c
Evaluated: m_05b52e
Evaluated: m_34c315
Evaluated: m_e6bed4
Evaluated: m_db2018
Evaluated: m_75dece
Evaluated: m_e1bfd4
Evaluated: m_e3b936

CASMI 2026 - Holdout Evaluation Results
---------------------------------------
               molecules  top1_matches  top25_matches  mrr_at_25
method                                                          
hybrid                30             1             22   0.113693
mass_only             30             0      

In [21]:
# CASMI 2026 - Compare spectral and hybrid rankings

comparison_30 = holdout_results_df.pivot(
    index="molecule_id",
    columns="method",
    values="known_rank"
).reset_index()

spectral_top25 = comparison_30["spectral_only"] <= 25
hybrid_top25 = comparison_30["hybrid"] <= 25

print("CASMI 2026 - Spectral vs Hybrid")
print("--------------------------------")

print(
    "Inside Top 25 with both methods:",
    int((spectral_top25 & hybrid_top25).sum())
)

print(
    "Found by spectral only:",
    int((spectral_top25 & ~hybrid_top25).sum())
)

print(
    "Found by hybrid only:",
    int((~spectral_top25 & hybrid_top25).sum())
)

print(
    "Missed by both methods:",
    int((~spectral_top25 & ~hybrid_top25).sum())
)

print("\nMolecules ranked first by spectral-only:")

print(
    comparison_30.loc[
        comparison_30["spectral_only"] == 1,
        [
            "molecule_id",
            "mass_only",
            "spectral_only",
            "hybrid"
        ]
    ].to_string(index=False)
)

CASMI 2026 - Spectral vs Hybrid
--------------------------------
Inside Top 25 with both methods: 18
Found by spectral only: 1
Found by hybrid only: 4
Missed by both methods: 7

Molecules ranked first by spectral-only:
molecule_id  mass_only  spectral_only  hybrid
   m_02d188         84              1      13
   m_05b52e         13              1       3
   m_0c4e10        110              1      19
   m_34c315         66              1      13
   m_3db789         56              1      15
   m_3dc915         41              1      13
   m_444a56         33              1       7
   m_57bec0         75              1      12
   m_89a6f2         20              1       9
   m_ace790          9              1       5
   m_c8934f        130              1      16
   m_d274b6          6              1       1
   m_db2018         16              1       9
   m_e1bfd4         18              1       9


In [22]:
# CASMI 2026 - Diagnose spectral evidence for holdout molecules

import pandas as pd

# Number of retained reference spectra for each structure and adduct
reference_counts = (
    holdout_reference_metadata
    .groupby(["inchikey14", "adduct"])
    .size()
)

diagnostic_rows = []

for molecule_id in holdout_ids:
    correct_key = holdout_label_df.loc[
        holdout_label_df["molecule_id"] == molecule_id,
        "inchikey14"
    ].iloc[0]

    query_adducts = set(
        holdout_queries.loc[
            holdout_queries["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    matching_references = sum(
        int(reference_counts.get((correct_key, adduct), 0))
        for adduct in query_adducts
    )

    molecule_ranks = comparison_30.loc[
        comparison_30["molecule_id"] == molecule_id
    ].iloc[0]

    diagnostic_rows.append({
        "molecule_id": molecule_id,
        "correct_structure_references": matching_references,
        "mass_rank": molecule_ranks["mass_only"],
        "spectral_rank": molecule_ranks["spectral_only"],
        "hybrid_rank": molecule_ranks["hybrid"]
    })

evidence_diagnostic = pd.DataFrame(diagnostic_rows)

print("CASMI 2026 - Spectral Evidence Diagnostic")
print("-----------------------------------------")
print("Holdout molecules:", len(evidence_diagnostic))
print(
    "Correct structures with matching reference spectra:",
    int(
        (
            evidence_diagnostic["correct_structure_references"] > 0
        ).sum()
    )
)
print(
    "Correct structures without matching reference spectra:",
    int(
        (
            evidence_diagnostic["correct_structure_references"] == 0
        ).sum()
    )
)

print("\nSpectral-only Top 1: reference evidence")
print(
    evidence_diagnostic.loc[
        evidence_diagnostic["spectral_rank"] == 1,
        [
            "molecule_id",
            "correct_structure_references",
            "mass_rank",
            "hybrid_rank"
        ]
    ].to_string(index=False)
)

print("\nMolecules with no matching reference spectra")
print(
    evidence_diagnostic.loc[
        evidence_diagnostic["correct_structure_references"] == 0,
        [
            "molecule_id",
            "mass_rank",
            "spectral_rank",
            "hybrid_rank"
        ]
    ].to_string(index=False)
)

CASMI 2026 - Spectral Evidence Diagnostic
-----------------------------------------
Holdout molecules: 30
Correct structures with matching reference spectra: 19
Correct structures without matching reference spectra: 11

Spectral-only Top 1: reference evidence
molecule_id  correct_structure_references  mass_rank  hybrid_rank
   m_3db789                             4         56           15
   m_3dc915                             3         41           13
   m_ace790                             5          9            5
   m_0c4e10                             3        110           19
   m_d274b6                             2          6            1
   m_02d188                             2         84           13
   m_89a6f2                             3         20            9
   m_444a56                             3         33            7
   m_57bec0                             2         75           12
   m_c8934f                             3        130           16
   m_05b52e   

In [23]:
# CASMI 2026 - Prepare structure-disjoint holdout references

import numpy as np

# These labels are used ONLY to construct an evaluation experiment.
# They must not be used when producing competition predictions.
holdout_correct_keys = set(
    holdout_label_df["inchikey14"]
)

# Remove every reference spectrum belonging to a correct
# holdout structure, not just exact copies of test spectra.
keep_mask = ~holdout_reference_metadata[
    "inchikey14"
].isin(holdout_correct_keys).to_numpy()

structure_disjoint_reference_features = (
    holdout_reference_features[keep_mask]
)

structure_disjoint_reference_metadata = (
    holdout_reference_metadata.loc[keep_mask]
    .reset_index(drop=True)
)

removed_count = int((~keep_mask).sum())

print("CASMI 2026 - Structure-Disjoint Preparation")
print("------------------------------------------")
print("Holdout molecules:", len(holdout_ids))
print("Correct structures excluded:", len(holdout_correct_keys))
print("Reference spectra before:", len(holdout_reference_metadata))
print("Reference spectra removed:", removed_count)
print(
    "Reference spectra remaining:",
    len(structure_disjoint_reference_metadata)
)
print(
    "Remaining feature matrix shape:",
    structure_disjoint_reference_features.shape
)

remaining_correct_refs = (
    structure_disjoint_reference_metadata["inchikey14"]
    .isin(holdout_correct_keys)
    .sum()
)

print(
    "Correct-structure reference spectra remaining:",
    int(remaining_correct_refs)
)

assert remaining_correct_refs == 0
assert (
    len(structure_disjoint_reference_features)
    == len(structure_disjoint_reference_metadata)
)

print("\nStructure-disjoint references prepared successfully!")

CASMI 2026 - Structure-Disjoint Preparation
------------------------------------------
Holdout molecules: 30
Correct structures excluded: 30
Reference spectra before: 27157
Reference spectra removed: 54
Reference spectra remaining: 27103
Remaining feature matrix shape: (27103, 1000)
Correct-structure reference spectra remaining: 0

Structure-disjoint references prepared successfully!


In [24]:
# CASMI 2026 - Structure-disjoint ranking evaluation

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

ref_keys = structure_disjoint_reference_metadata[
    "inchikey14"
].to_numpy()

ref_adducts = structure_disjoint_reference_metadata[
    "adduct"
].to_numpy()

correct_keys = holdout_label_df.set_index(
    "molecule_id"
)["inchikey14"]

disjoint_rows = []

print("CASMI 2026 - Structure-Disjoint Evaluation")
print("------------------------------------------")

for molecule_id in holdout_ids:

    queries = holdout_queries[
        holdout_queries["molecule_id"] == molecule_id
    ]

    query_mass = float(mass_lookup.loc[molecule_id])
    candidate_keys = holdout_candidates[molecule_id]

    candidates = library_with_mass[
        library_with_mass["inchikey14"].isin(candidate_keys)
    ][["inchikey14", "reference_mass"]].copy()

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    score_sums = {key: 0.0 for key in candidate_keys}
    score_counts = {key: 0 for key in candidate_keys}

    candidate_mask = np.isin(
        ref_keys, list(candidate_keys)
    )

    # Match each query only against references with the same adduct.
    for adduct, query_group in queries.groupby("adduct"):

        ref_positions = np.flatnonzero(
            candidate_mask & (ref_adducts == adduct)
        )

        if len(ref_positions) == 0:
            continue

        X_query = np.stack([
            make_feature_vector(row)
            for row in query_group.itertuples(index=False)
        ])

        X_ref = structure_disjoint_reference_features[
            ref_positions
        ]

        similarities = cosine_similarity(X_query, X_ref)
        selected_keys = ref_keys[ref_positions]

        for key in np.unique(selected_keys):

            positions = np.flatnonzero(
                selected_keys == key
            )

            best_scores = similarities[
                :, positions
            ].max(axis=1)

            score_sums[key] += float(best_scores.sum())
            score_counts[key] += len(best_scores)

    # Preserve the scoring method from the previous evaluation.
    candidates["spectral_score"] = candidates[
        "inchikey14"
    ].map(
        lambda key: (
            score_sums[key] / len(queries)
            if score_counts[key] > 0
            else np.nan
        )
    )

    mass_order = candidates.sort_values(
        ["mass_error_ppm", "inchikey14"],
        ascending=[True, True]
    ).reset_index(drop=True)

    mass_ranks = {
        key: rank
        for rank, key in enumerate(
            mass_order["inchikey14"], start=1
        )
    }

    spectral_order = candidates.sort_values(
        ["spectral_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    spectral_ranks = {
        key: rank
        for rank, key in enumerate(
            spectral_order.dropna(
                subset=["spectral_score"]
            )["inchikey14"],
            start=1
        )
    }

    def hybrid_score(key):
        mass_rank = mass_ranks[key]
        spectral_rank = spectral_ranks.get(
            key, mass_rank
        )

        return (
            0.7 / (10 + mass_rank)
            + 0.3 / (10 + spectral_rank)
        )

    candidates["hybrid_score"] = candidates[
        "inchikey14"
    ].map(hybrid_score)

    hybrid_order = candidates.sort_values(
        ["hybrid_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    # Labels are used only after rankings have been produced.
    known_key = correct_keys.loc[molecule_id]

    for method, ordered in [
        ("mass_only", mass_order),
        ("spectral_only", spectral_order),
        ("hybrid", hybrid_order)
    ]:

        positions = np.flatnonzero(
            ordered["inchikey14"].to_numpy() == known_key
        )

        rank = (
            int(positions[0]) + 1
            if len(positions) == 1
            else None
        )

        disjoint_rows.append({
            "molecule_id": molecule_id,
            "method": method,
            "known_rank": rank,
            "top1": rank == 1,
            "top25": rank is not None and rank <= 25,
            "reciprocal_rank_at_25": (
                1.0 / rank
                if rank is not None and rank <= 25
                else 0.0
            )
        })

    print("Evaluated:", molecule_id)


disjoint_results_df = pd.DataFrame(disjoint_rows)

disjoint_summary = (
    disjoint_results_df.groupby("method")
    .agg(
        molecules=("molecule_id", "count"),
        top1_matches=("top1", "sum"),
        top25_matches=("top25", "sum"),
        mrr_at_25=("reciprocal_rank_at_25", "mean")
    )
)

print("\nCASMI 2026 - Structure-Disjoint Results")
print("---------------------------------------")
print(disjoint_summary.to_string())

print("\nOriginal holdout results for comparison:")
print(holdout_summary.to_string())

CASMI 2026 - Structure-Disjoint Evaluation
------------------------------------------
Evaluated: m_cfb3df
Evaluated: m_3db789
Evaluated: m_fea652
Evaluated: m_370436
Evaluated: m_3dc915
Evaluated: m_53dfe2
Evaluated: m_2aa8ea
Evaluated: m_ace790
Evaluated: m_e993ec
Evaluated: m_0c4e10
Evaluated: m_cd50d8
Evaluated: m_d56a4e
Evaluated: m_762dcf
Evaluated: m_d274b6
Evaluated: m_2b7513
Evaluated: m_02d188
Evaluated: m_95e8da
Evaluated: m_43e809
Evaluated: m_89a6f2
Evaluated: m_444a56
Evaluated: m_57bec0
Evaluated: m_c8934f
Evaluated: m_6f3a5c
Evaluated: m_05b52e
Evaluated: m_34c315
Evaluated: m_e6bed4
Evaluated: m_db2018
Evaluated: m_75dece
Evaluated: m_e1bfd4
Evaluated: m_e3b936

CASMI 2026 - Structure-Disjoint Results
---------------------------------------
               molecules  top1_matches  top25_matches  mrr_at_25
method                                                          
hybrid                30             0             12   0.044406
mass_only             30             0

In [25]:
# CASMI 2026 - Check molecular structure tools

import rdkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

print("CASMI 2026 - Molecular Structure Tools")
print("--------------------------------------")
print("RDKit version:", rdkit.__version__)

example_smiles = library_with_mass[
    "normalized_smiles"
].dropna().iloc[0]

molecule = Chem.MolFromSmiles(example_smiles)

if molecule is None:
    print("SMILES parsing failed:", example_smiles)
else:
    print("SMILES parsing: successful")
    print("Example SMILES:", example_smiles)
    print(
        "Molecular formula:",
        rdMolDescriptors.CalcMolFormula(molecule)
    )
    print("Number of atoms:", molecule.GetNumAtoms())
    print("Number of bonds:", molecule.GetNumBonds())

ModuleNotFoundError: No module named 'rdkit'

In [26]:
%pip install -q rdkit

ERROR: Could not find a version that satisfies the requirement rdkit (from versions: none)
ERROR: No matching distribution found for rdkit
Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip install -q rdkit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 42.3 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [28]:
import rdkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

example_smiles = library_with_mass[
    "normalized_smiles"
].dropna().iloc[0]

molecule = Chem.MolFromSmiles(example_smiles)

print("CASMI 2026 - RDKit Check")
print("-----------------------")
print("RDKit version:", rdkit.__version__)
print("SMILES parsing successful:", molecule is not None)

if molecule is not None:
    print("Example SMILES:", example_smiles)
    print(
        "Molecular formula:",
        rdMolDescriptors.CalcMolFormula(molecule)
    )
    print("Number of atoms:", molecule.GetNumAtoms())
    print("Number of bonds:", molecule.GetNumBonds())

CASMI 2026 - RDKit Check
-----------------------
RDKit version: 2026.03.6
SMILES parsing successful: True
Example SMILES: O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21
Molecular formula: C20H15N3O2
Number of atoms: 25
Number of bonds: 28


In [29]:
# CASMI 2026 - Build RDKit features for holdout candidates

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

candidate_structures = (
    library_with_mass.loc[
        library_with_mass["inchikey14"].isin(holdout_candidate_keys),
        ["inchikey14", "normalized_smiles",
         "molecular_formula", "reference_mass"]
    ]
    .drop_duplicates(subset=["inchikey14"])
    .copy()
)

structure_rows = []
failed_keys = []

for row in candidate_structures.itertuples(index=False):
    mol = Chem.MolFromSmiles(row.normalized_smiles)

    if mol is None:
        failed_keys.append(row.inchikey14)
        continue

    structure_rows.append({
        "inchikey14": row.inchikey14,
        "rdkit_formula": rdMolDescriptors.CalcMolFormula(mol),
        "rdkit_exact_mass": rdMolDescriptors.CalcExactMolWt(mol),
        "atom_count": mol.GetNumAtoms(),
        "bond_count": mol.GetNumBonds(),
        "library_formula": row.molecular_formula,
        "library_reference_mass": row.reference_mass
    })

holdout_structure_features = pd.DataFrame(structure_rows)

holdout_structure_features["mass_difference_da"] = (
    holdout_structure_features["rdkit_exact_mass"]
    - holdout_structure_features["library_reference_mass"]
).abs()

formula_matches = (
    holdout_structure_features["rdkit_formula"]
    == holdout_structure_features["library_formula"]
)

print("CASMI 2026 - RDKit Candidate Features")
print("-------------------------------------")
print("Unique candidates:", len(candidate_structures))
print("Successfully parsed:", len(holdout_structure_features))
print("Failed to parse:", len(failed_keys))
print("Matching molecular formulas:", int(formula_matches.sum()))
print("Different molecular formulas:", int((~formula_matches).sum()))
print(
    "Median exact-mass difference (Da):",
    round(
        holdout_structure_features["mass_difference_da"].median(),
        6
    )
)
print(
    "Maximum exact-mass difference (Da):",
    round(
        holdout_structure_features["mass_difference_da"].max(),
        6
    )
)

print("\nExample structure features:")
print(
    holdout_structure_features[
        ["rdkit_formula", "rdkit_exact_mass",
         "atom_count", "bond_count"]
    ].head(5).to_string(index=False)
)

print("\nRDKit candidate feature extraction completed!")

CASMI 2026 - RDKit Candidate Features
-------------------------------------
Unique candidates: 5172
Successfully parsed: 5172
Failed to parse: 0
Matching molecular formulas: 5172
Different molecular formulas: 0
Median exact-mass difference (Da): 0.0
Maximum exact-mass difference (Da): 0.000549

Example structure features:
rdkit_formula  rdkit_exact_mass  atom_count  bond_count
  C21H21N3O3S        395.130363          28          30
 C14H22BrN3O2        343.089539          20          20
   C10H13N5O5        283.091669          20          22
  C21H25FN2O2        356.190006          26          28
 C17H23ClN2O3        338.139720          23          24

RDKit candidate feature extraction completed!


In [30]:
# CASMI 2026 - Diagnose RDKit exact-mass differences

import numpy as np
from rdkit import Chem

mass_mismatches = holdout_structure_features.loc[
    holdout_structure_features["mass_difference_da"] > 0.0001
].copy()

smiles_lookup = candidate_structures.set_index(
    "inchikey14"
)["normalized_smiles"]

if not mass_mismatches.empty:
    mass_mismatches["formal_charge"] = (
        mass_mismatches["inchikey14"].map(
            lambda key: Chem.GetFormalCharge(
                Chem.MolFromSmiles(smiles_lookup.loc[key])
            )
        )
    )

print("CASMI 2026 - Exact-Mass Diagnostic")
print("----------------------------------")
print("Candidates checked:", len(holdout_structure_features))
print("Differences above 0.0001 Da:", len(mass_mismatches))

if not mass_mismatches.empty:
    print("\nFormal charges among mismatched candidates:")
    print(
        mass_mismatches["formal_charge"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nFirst 10 mismatched candidates:")
    print(
        mass_mismatches[
            [
                "rdkit_formula",
                "library_formula",
                "formal_charge",
                "mass_difference_da"
            ]
        ]
        .head(10)
        .to_string(index=False)
    )
else:
    print("\nNo significant mass differences found.")

CASMI 2026 - Exact-Mass Diagnostic
----------------------------------
Candidates checked: 5172
Differences above 0.0001 Da: 10

Formal charges among mismatched candidates:
formal_charge
1    10

First 10 mismatched candidates:
rdkit_formula library_formula  formal_charge  mass_difference_da
   C20H20NO4+      C20H20NO4+              1            0.000549
   C21H26NO4+      C21H26NO4+              1            0.000549
   C20H20NO4+      C20H20NO4+              1            0.000549
   C21H26NO4+      C21H26NO4+              1            0.000549
   C20H20NO4+      C20H20NO4+              1            0.000549
   C20H20NO4+      C20H20NO4+              1            0.000549
   C20H20NO4+      C20H20NO4+              1            0.000549
   C21H26NO4+      C21H26NO4+              1            0.000549
   C20H20NO4+      C20H20NO4+              1            0.000549
   C20H20NO4+      C20H20NO4+              1            0.000549


In [31]:
# CASMI 2026 - Account for electron mass in charged formulas

import numpy as np
from rdkit import Chem

ELECTRON_MASS_DA = 0.000548579909065

smiles_by_key = candidate_structures.set_index(
    "inchikey14"
)["normalized_smiles"]

holdout_structure_features["formal_charge"] = (
    holdout_structure_features["inchikey14"].map(
        lambda key: Chem.GetFormalCharge(
            Chem.MolFromSmiles(smiles_by_key.loc[key])
        )
    )
)

# The original library mass is the sum of neutral atomic masses.
# A +1 ion has one fewer electron; a -1 ion has one extra electron.
holdout_structure_features["charge_corrected_mass"] = (
    holdout_structure_features["library_reference_mass"]
    - holdout_structure_features["formal_charge"]
    * ELECTRON_MASS_DA
)

holdout_structure_features["corrected_difference_da"] = (
    holdout_structure_features["rdkit_exact_mass"]
    - holdout_structure_features["charge_corrected_mass"]
).abs()

print("CASMI 2026 - Charge-Corrected Mass Check")
print("----------------------------------------")
print("Candidates checked:", len(holdout_structure_features))

print("\nFormal charge distribution:")
print(
    holdout_structure_features["formal_charge"]
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\nDifferences above 0.0001 Da before correction:",
    int(
        (
            holdout_structure_features["mass_difference_da"]
            > 0.0001
        ).sum()
    )
)

print(
    "Differences above 0.0001 Da after correction:",
    int(
        (
            holdout_structure_features["corrected_difference_da"]
            > 0.0001
        ).sum()
    )
)

print(
    "Maximum difference after correction (Da):",
    round(
        float(
            holdout_structure_features[
                "corrected_difference_da"
            ].max()
        ),
        9
    )
)

print("\nSubmission mass filter unchanged.")

CASMI 2026 - Charge-Corrected Mass Check
----------------------------------------
Candidates checked: 5172

Formal charge distribution:
formal_charge
0    5162
1      10

Differences above 0.0001 Da before correction: 10
Differences above 0.0001 Da after correction: 0
Maximum difference after correction (Da): 5.2e-07

Submission mass filter unchanged.


In [32]:
# CASMI 2026 - Inspect experimental fragment peaks

import numpy as np
import pandas as pd

# Use the original development subset, NOT the 30-molecule holdout.
development_spectra = evaluation_queries[
    evaluation_queries["adduct"] == "[M+H]+"
].copy()

assert not development_spectra.empty, (
    "No [M+H]+ spectrum found in the development subset."
)

# Choose one reproducible development spectrum.
query = development_spectra.sort_values(
    ["molecule_id", "spectrum_id"]
).iloc[0]

# Precursor m/z was not included in the compact test_spectra table.
precursor_lookup = (
    test_file.read(
        columns=["spectrum_id", "precursor_mz"]
    )
    .to_pandas()
    .set_index("spectrum_id")["precursor_mz"]
)

precursor_mz = float(
    precursor_lookup.loc[query["spectrum_id"]]
)

mzs = np.asarray(query["ms2_mzs"], dtype=float)
intensities = np.asarray(
    query["ms2_normalized_intensities"],
    dtype=float
)

valid = (
    np.isfinite(mzs)
    & np.isfinite(intensities)
    & (mzs > 0)
    & (intensities > 0)
)

peaks = pd.DataFrame({
    "fragment_mz": mzs[valid],
    "intensity": intensities[valid]
})

# For a [M+H]+ spectrum, this is the observed difference
# between the precursor ion and each fragment peak.
peaks["neutral_loss_da"] = (
    precursor_mz - peaks["fragment_mz"]
)

top_peaks = (
    peaks.sort_values(
        "intensity", ascending=False
    )
    .head(12)
    .copy()
)

print("CASMI 2026 - Fragment Peak Inspection")
print("-------------------------------------")
print("Development molecule:", query["molecule_id"])
print("Spectrum:", query["spectrum_id"])
print("Adduct:", query["adduct"])
print("Precursor m/z:", round(precursor_mz, 6))
print("Total valid fragment peaks:", len(peaks))

print("\n12 most intense experimental peaks:")
print(
    top_peaks.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

CASMI 2026 - Fragment Peak Inspection
-------------------------------------
Development molecule: m_145679
Spectrum: s_112ad6b0
Adduct: [M+H]+
Precursor m/z: 320.1608
Total valid fragment peaks: 42

12 most intense experimental peaks:
 fragment_mz  intensity  neutral_loss_da
   86.060300   1.000000       234.100500
   81.033600   0.448799       239.127200
  113.071800   0.318452       207.089000
   87.063600   0.070259       233.097200
  114.075200   0.022138       206.085600
   82.037300   0.018994       238.123500
   71.023600   0.015739       249.137200
   68.049700   0.015073       252.111100
   95.050600   0.010264       225.110200
   71.050300   0.006917       249.110500
   71.047600   0.006861       249.113200
   68.052700   0.005104       252.108100


In [33]:
# CASMI 2026 - Inspect the known development structure

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

development_id = query["molecule_id"]
correct_key = labels.loc[development_id]

structure = library_with_mass.loc[
    library_with_mass["inchikey14"] == correct_key
].iloc[0]

mol = Chem.MolFromSmiles(structure["normalized_smiles"])
assert mol is not None, "RDKit could not parse the known SMILES."

formula = rdMolDescriptors.CalcMolFormula(mol)
exact_mass = rdMolDescriptors.CalcExactMolWt(mol)
expected_precursor = exact_mass + 1.007276466621

print("CASMI 2026 - Development Structure")
print("----------------------------------")
print("Molecule ID:", development_id)
print("Known structure key:", correct_key)
print("SMILES:", structure["normalized_smiles"])
print("Molecular formula:", formula)
print("Exact molecular mass:", round(exact_mass, 6))
print("Observed precursor m/z:", round(precursor_mz, 6))
print(
    "Expected [M+H]+ m/z:",
    round(expected_precursor, 6)
)
print(
    "Precursor difference (Da):",
    round(precursor_mz - expected_precursor, 6)
)
print("Atoms:", mol.GetNumAtoms())
print("Bonds:", mol.GetNumBonds())
print("Rings:", mol.GetRingInfo().NumRings())

CASMI 2026 - Development Structure
----------------------------------
Molecule ID: m_145679
Known structure key: XHTLRVHKUGTNCU
SMILES: CC(C)c1nnc(C2COCCN2C(=O)CCc2ccco2)o1
Molecular formula: C16H21N3O4
Exact molecular mass: 319.153206
Observed precursor m/z: 320.1608
Expected [M+H]+ m/z: 320.160483
Precursor difference (Da): 0.000317
Atoms: 23
Bonds: 25
Rings: 3


In [34]:
# CASMI 2026 - Match dominant peaks to possible fragment formulas

from itertools import product
from collections import Counter
from rdkit import Chem
import pandas as pd

# Count all atoms, including implicit hydrogens, in the known
# development structure. Add one proton for the [M+H]+ precursor.
molecule_with_h = Chem.AddHs(mol)

precursor_atoms = Counter(
    atom.GetSymbol()
    for atom in molecule_with_h.GetAtoms()
)
precursor_atoms["H"] += 1

assert set(precursor_atoms).issubset({"C", "H", "N", "O"})

atomic_mass = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620
}

electron_mass = 0.000548579909065
mass_tolerance_ppm = 20.0

# Enumerate elemental compositions that do not exceed the
# protonated precursor's atom counts.
possible_formulas = []

for c, h, n, o in product(
    range(precursor_atoms["C"] + 1),
    range(precursor_atoms["H"] + 1),
    range(precursor_atoms["N"] + 1),
    range(precursor_atoms["O"] + 1)
):
    if c + n + o == 0:
        continue

    # A singly charged positive ion has one electron fewer
    # than the sum of its constituent neutral atoms.
    theoretical_mz = (
        c * atomic_mass["C"]
        + h * atomic_mass["H"]
        + n * atomic_mass["N"]
        + o * atomic_mass["O"]
        - electron_mass
    )

    formula = "".join(
        f"{element}{count if count > 1 else ''}"
        for element, count in (
            ("C", c), ("H", h), ("N", n), ("O", o)
        )
        if count > 0
    ) + "+"

    possible_formulas.append((formula, theoretical_mz))

matches = []

for peak in top_peaks.head(3).itertuples(index=False):

    for formula, theoretical_mz in possible_formulas:

        error_ppm = (
            (peak.fragment_mz - theoretical_mz)
            / theoretical_mz
            * 1_000_000
        )

        if abs(error_ppm) <= mass_tolerance_ppm:
            matches.append({
                "observed_mz": peak.fragment_mz,
                "intensity": peak.intensity,
                "possible_formula": formula,
                "theoretical_mz": theoretical_mz,
                "mass_error_ppm": error_ppm
            })

fragment_formula_matches = pd.DataFrame(matches)

print("CASMI 2026 - Fragment Formula Matching")
print("--------------------------------------")
print("Development molecule:", development_id)
print("Precursor composition:", dict(precursor_atoms))
print("Possible formulas checked:", len(possible_formulas))
print("Mass tolerance:", mass_tolerance_ppm, "ppm")

if fragment_formula_matches.empty:
    print("\nNo matching formulas found for the three peaks.")
else:
    fragment_formula_matches["absolute_error_ppm"] = (
        fragment_formula_matches["mass_error_ppm"].abs()
    )

    print("\nPossible formulas for the three strongest peaks:")

    for observed_mz, group in fragment_formula_matches.groupby(
        "observed_mz", sort=False
    ):
        print(f"\nObserved peak: {observed_mz:.4f} m/z")
        print(
            group.sort_values("absolute_error_ppm")[
                [
                    "possible_formula",
                    "theoretical_mz",
                    "mass_error_ppm"
                ]
            ].head(5).to_string(
                index=False,
                float_format=lambda x: f"{x:.6f}"
            )
        )

CASMI 2026 - Fragment Formula Matching
--------------------------------------
Development molecule: m_145679
Precursor composition: {'C': 16, 'N': 3, 'O': 4, 'H': 22}
Possible formulas checked: 7797
Mass tolerance: 20.0 ppm

Possible formulas for the three strongest peaks:

Observed peak: 86.0603 m/z
possible_formula  theoretical_mz  mass_error_ppm
         C4H8NO+       86.060040        3.017660

Observed peak: 81.0336 m/z
possible_formula  theoretical_mz  mass_error_ppm
          C5H5O+       81.033491        1.342654
         C3H3N3+       81.032149       17.912297

Observed peak: 113.0718 m/z
possible_formula  theoretical_mz  mass_error_ppm
        C5H9N2O+      113.070939        7.611716


In [35]:
# CASMI 2026 - Test structural plausibility of fragment formulas

from itertools import combinations
from collections import Counter
from rdkit import Chem
import pandas as pd

# Known development molecule from our previous step
mol_with_h = Chem.AddHs(mol)

heavy_bonds = [
    (
        bond.GetBeginAtomIdx(),
        bond.GetEndAtomIdx()
    )
    for bond in mol.GetBonds()
]

targets = {
    "86.0603": {"C": 4, "H": 8, "N": 1, "O": 1},
    "81.0336": {"C": 5, "H": 5, "N": 0, "O": 1},
    "113.0718": {"C": 5, "H": 9, "N": 2, "O": 1}
}

structural_hits = []
cut_sets_checked = 0

# Try breaking one or two heavy-atom bonds.
for number_of_cuts in (1, 2):

    for bonds_to_cut in combinations(
        heavy_bonds, number_of_cuts
    ):
        cut_sets_checked += 1

        editable = Chem.RWMol(mol_with_h)

        for atom_a, atom_b in bonds_to_cut:
            editable.RemoveBond(atom_a, atom_b)

        fragments = Chem.GetMolFrags(
            editable.GetMol(),
            asMols=False
        )

        # A cut that does not separate the graph is not
        # counted as an isolated fragment.
        if len(fragments) < 2:
            continue

        for fragment_indices in fragments:

            atom_counts = Counter(
                mol_with_h.GetAtomWithIdx(atom_idx)
                .GetSymbol()
                for atom_idx in fragment_indices
            )

            if sum(
                atom_counts[element]
                for element in ("C", "N", "O")
            ) < 2:
                continue

            for peak, target in targets.items():

                # First compare heavy-atom composition.
                if not all(
                    atom_counts[element] == target[element]
                    for element in ("C", "N", "O")
                ):
                    continue

                hydrogen_difference = (
                    target["H"] - atom_counts["H"]
                )

                structural_hits.append({
                    "peak_mz": peak,
                    "bonds_cut": number_of_cuts,
                    "hydrogen_difference": hydrogen_difference,
                    "cut_atom_pairs": str(bonds_to_cut)
                })

structural_hits_df = pd.DataFrame(structural_hits)

print("CASMI 2026 - Structural Fragment Check")
print("-------------------------------------")
print("Development molecule:", development_id)
print("Heavy-atom bonds:", len(heavy_bonds))
print("Bond-cut combinations checked:", cut_sets_checked)

for peak in targets:

    print(f"\nPeak: {peak} m/z")

    if structural_hits_df.empty:
        print("No matching heavy-atom fragment found.")
        continue

    matches = structural_hits_df[
        structural_hits_df["peak_mz"] == peak
    ].copy()

    if matches.empty:
        print(
            "No matching heavy-atom fragment found "
            "using up to two bond cuts."
        )
        continue

    matches["absolute_h_difference"] = (
        matches["hydrogen_difference"].abs()
    )

    print("Matching connected fragments:", len(matches))
    print(
        matches.sort_values(
            ["bonds_cut", "absolute_h_difference"]
        )[
            [
                "bonds_cut",
                "hydrogen_difference",
                "cut_atom_pairs"
            ]
        ].head(5).to_string(index=False)
    )

CASMI 2026 - Structural Fragment Check
-------------------------------------
Development molecule: m_145679
Heavy-atom bonds: 25
Bond-cut combinations checked: 325

Peak: 86.0603 m/z
Matching connected fragments: 2
 bonds_cut  hydrogen_difference     cut_atom_pairs
         2                    1  ((4, 5), (6, 22))
         2                    1 ((6, 7), (12, 13))

Peak: 81.0336 m/z
Matching connected fragments: 24
 bonds_cut  hydrogen_difference     cut_atom_pairs
         1                    0        ((15, 16),)
         2                    0 ((0, 1), (15, 16))
         2                    0 ((1, 2), (15, 16))
         2                    0 ((1, 3), (15, 16))
         2                    0 ((3, 4), (15, 16))

Peak: 113.0718 m/z
Matching connected fragments: 22
 bonds_cut  hydrogen_difference   cut_atom_pairs
         1                    2        ((6, 7),)
         2                    2 ((3, 4), (6, 7))
         2                    2 ((4, 5), (6, 7))
         2               

In [36]:
# CASMI 2026 - Compare one-bond fragment matches across candidates

from collections import Counter
from rdkit import Chem
import pandas as pd

candidate_keys = evaluation_candidates[development_id]

development_candidates = (
    library_with_mass.loc[
        library_with_mass["inchikey14"].isin(candidate_keys),
        ["inchikey14", "normalized_smiles"]
    ]
    .drop_duplicates("inchikey14")
    .copy()
)

target_fragments = {
    "86.0603": {"C": 4, "H": 8, "N": 1, "O": 1},
    "81.0336": {"C": 5, "H": 5, "N": 0, "O": 1},
    "113.0718": {"C": 5, "H": 9, "N": 2, "O": 1},
}

def one_bond_fragment_matches(smiles, targets):
    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        return None

    molecule_with_h = Chem.AddHs(molecule)
    matched_peaks = set()

    for bond in molecule.GetBonds():
        editable = Chem.RWMol(molecule_with_h)

        editable.RemoveBond(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx()
        )

        fragments = Chem.GetMolFrags(
            editable.GetMol(),
            asMols=False
        )

        # Breaking a ring bond may leave the molecule connected.
        if len(fragments) < 2:
            continue

        for atom_indices in fragments:
            counts = Counter(
                molecule_with_h
                .GetAtomWithIdx(index)
                .GetSymbol()
                for index in atom_indices
            )

            for peak, target in targets.items():

                heavy_atoms_match = all(
                    counts[element] == target[element]
                    for element in ("C", "N", "O")
                )

                hydrogen_difference = (
                    target["H"] - counts["H"]
                )

                if (
                    heavy_atoms_match
                    and abs(hydrogen_difference) <= 2
                ):
                    matched_peaks.add(peak)

    return matched_peaks


candidate_fragment_rows = []

for row in development_candidates.itertuples(index=False):

    matched = one_bond_fragment_matches(
        row.normalized_smiles,
        target_fragments
    )

    candidate_fragment_rows.append({
        "inchikey14": row.inchikey14,
        "parsed": matched is not None,
        "match_86": matched is not None
            and "86.0603" in matched,
        "match_81": matched is not None
            and "81.0336" in matched,
        "match_113": matched is not None
            and "113.0718" in matched
    })

candidate_fragment_df = pd.DataFrame(candidate_fragment_rows)

print("CASMI 2026 - Candidate Fragment Comparison")
print("------------------------------------------")
print("Development molecule:", development_id)
print("Candidates checked:", len(candidate_fragment_df))
print(
    "Successfully parsed:",
    int(candidate_fragment_df["parsed"].sum())
)

print("\nCandidates matching each peak with one bond cut:")

for column, peak in (
    ("match_86", "86.0603"),
    ("match_81", "81.0336"),
    ("match_113", "113.0718")
):
    print(
        f"{peak} m/z:",
        int(candidate_fragment_df[column].sum())
    )

known_row = candidate_fragment_df.loc[
    candidate_fragment_df["inchikey14"] == correct_key
]

assert len(known_row) == 1

print("\nKnown development structure:")
print(
    known_row[
        ["match_86", "match_81", "match_113"]
    ].to_string(index=False)
)

CASMI 2026 - Candidate Fragment Comparison
------------------------------------------
Development molecule: m_145679
Candidates checked: 181
Successfully parsed: 181

Candidates matching each peak with one bond cut:
86.0603 m/z: 19
81.0336 m/z: 6
113.0718 m/z: 10

Known development structure:
 match_86  match_81  match_113
    False      True       True


In [37]:
# CASMI 2026 - Test one-bond fragment ranking on one development molecule

import numpy as np
import pandas as pd

# Use the observed intensities of the three strongest peaks.
peak_weights = {
    f"{row.fragment_mz:.4f}": float(row.intensity)
    for row in top_peaks.head(3).itertuples(index=False)
}

ranking_test = candidate_fragment_df.merge(
    library_with_mass[
        ["inchikey14", "reference_mass"]
    ],
    on="inchikey14",
    how="left",
    validate="one_to_one"
)

ranking_test["mass_error_ppm"] = (
    (
        ranking_test["reference_mass"]
        - float(mass_lookup.loc[development_id])
    ).abs()
    / float(mass_lookup.loc[development_id])
    * 1_000_000
)

# The score does not use the known structure label.
ranking_test["one_bond_score"] = (
    ranking_test["match_86"].astype(float)
    * peak_weights["86.0603"]
    + ranking_test["match_81"].astype(float)
    * peak_weights["81.0336"]
    + ranking_test["match_113"].astype(float)
    * peak_weights["113.0718"]
)

mass_order = ranking_test.sort_values(
    ["mass_error_ppm", "inchikey14"],
    ascending=[True, True]
).reset_index(drop=True)

fragment_order = ranking_test.sort_values(
    ["one_bond_score", "mass_error_ppm", "inchikey14"],
    ascending=[False, True, True]
).reset_index(drop=True)

# Evaluate the known structure only after both rankings are complete.
def get_known_rank(ordered, known_key):
    positions = np.flatnonzero(
        ordered["inchikey14"].to_numpy() == known_key
    )
    assert len(positions) == 1
    return int(positions[0]) + 1

mass_rank = get_known_rank(mass_order, correct_key)
fragment_rank = get_known_rank(fragment_order, correct_key)

known_score = ranking_test.loc[
    ranking_test["inchikey14"] == correct_key,
    "one_bond_score"
].iloc[0]

print("CASMI 2026 - One-Bond Ranking Diagnostic")
print("----------------------------------------")
print("Development molecule:", development_id)
print("Candidates evaluated:", len(ranking_test))
print("Mass-only known-structure rank:", mass_rank)
print("One-bond known-structure rank:", fragment_rank)
print("Known-structure fragment score:", round(float(known_score), 6))
print("Known structure in fragment Top 25:", fragment_rank <= 25)

print("\nTop 5 candidates by one-bond score:")
print(
    fragment_order[
        [
            "inchikey14",
            "one_bond_score",
            "mass_error_ppm",
            "match_86",
            "match_81",
            "match_113"
        ]
    ].head(5).to_string(index=False)
)

CASMI 2026 - One-Bond Ranking Diagnostic
----------------------------------------
Development molecule: m_145679
Candidates evaluated: 181
Mass-only known-structure rank: 75
One-bond known-structure rank: 20
Known-structure fragment score: 0.76725
Known structure in fragment Top 25: True

Top 5 candidates by one-bond score:
    inchikey14  one_bond_score  mass_error_ppm  match_86  match_81  match_113
XFXCMBIAVTWKDC        1.448799        0.681081      True      True      False
GVJKCZSPTZHHEA        1.318452        9.882183      True     False       True
ARBZLNAPSFGXRM        1.000000        0.681081      True     False      False
DSPQOFFIGDEUOT        1.000000        0.681081      True     False      False
FUCACFWCSWHQKP        1.000000        0.681081      True     False      False


In [38]:
# CASMI 2026 - Compare one-bond and up-to-two-bond fragment ranking

from itertools import combinations
from collections import Counter
from rdkit import Chem
import numpy as np
import pandas as pd

def two_bond_fragment_matches(smiles, targets):
    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        return None

    molecule_with_h = Chem.AddHs(molecule)
    bonds = [
        (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        for bond in molecule.GetBonds()
    ]

    matched_peaks = set()

    # Test exactly two bond cuts. The existing one-bond
    # results will be combined with these matches afterward.
    for bonds_to_cut in combinations(bonds, 2):
        editable = Chem.RWMol(molecule_with_h)

        for atom_a, atom_b in bonds_to_cut:
            editable.RemoveBond(atom_a, atom_b)

        fragments = Chem.GetMolFrags(
            editable.GetMol(),
            asMols=False
        )

        if len(fragments) < 2:
            continue

        for atom_indices in fragments:
            counts = Counter(
                molecule_with_h.GetAtomWithIdx(index).GetSymbol()
                for index in atom_indices
            )

            for peak, target in targets.items():
                heavy_atoms_match = all(
                    counts[element] == target[element]
                    for element in ("C", "N", "O")
                )

                hydrogen_difference = (
                    target["H"] - counts["H"]
                )

                if (
                    heavy_atoms_match
                    and abs(hydrogen_difference) <= 2
                ):
                    matched_peaks.add(peak)

        if len(matched_peaks) == len(targets):
            break

    return matched_peaks


two_bond_rows = []

for row in development_candidates.itertuples(index=False):
    matched = two_bond_fragment_matches(
        row.normalized_smiles,
        target_fragments
    )

    two_bond_rows.append({
        "inchikey14": row.inchikey14,
        "two_bond_86": matched is not None
            and "86.0603" in matched,
        "two_bond_81": matched is not None
            and "81.0336" in matched,
        "two_bond_113": matched is not None
            and "113.0718" in matched
    })

two_bond_df = pd.DataFrame(two_bond_rows)

# Keep the previous one-bond matches and add the new matches.
combined_ranking = ranking_test.merge(
    two_bond_df,
    on="inchikey14",
    how="left",
    validate="one_to_one"
)

for peak in ("86", "81", "113"):
    combined_ranking[f"up_to_two_{peak}"] = (
        combined_ranking[f"match_{peak}"]
        | combined_ranking[f"two_bond_{peak}"]
    )

# Count each observed peak once, even if several bond-cut
# combinations produce matching fragments.
combined_ranking["up_to_two_bond_score"] = (
    combined_ranking["up_to_two_86"].astype(float)
    * peak_weights["86.0603"]
    + combined_ranking["up_to_two_81"].astype(float)
    * peak_weights["81.0336"]
    + combined_ranking["up_to_two_113"].astype(float)
    * peak_weights["113.0718"]
)

two_bond_order = combined_ranking.sort_values(
    [
        "up_to_two_bond_score",
        "mass_error_ppm",
        "inchikey14"
    ],
    ascending=[False, True, True]
).reset_index(drop=True)

two_bond_rank = get_known_rank(
    two_bond_order,
    correct_key
)

known_two_bond_row = combined_ranking.loc[
    combined_ranking["inchikey14"] == correct_key
].iloc[0]

print("CASMI 2026 - Two-Bond Ranking Diagnostic")
print("----------------------------------------")
print("Development molecule:", development_id)
print("Candidates evaluated:", len(combined_ranking))
print("Mass-only known-structure rank:", mass_rank)
print("One-bond known-structure rank:", fragment_rank)
print("Up-to-two-bond known-structure rank:", two_bond_rank)

print("\nCandidates matching each peak:")
for peak in ("86", "81", "113"):
    print(
        f"{peak}: "
        f"one bond={int(combined_ranking[f'match_{peak}'].sum())}, "
        f"up to two bonds="
        f"{int(combined_ranking[f'up_to_two_{peak}'].sum())}"
    )

print("\nKnown structure matches:")
print(
    "86.0603:",
    bool(known_two_bond_row["up_to_two_86"])
)
print(
    "81.0336:",
    bool(known_two_bond_row["up_to_two_81"])
)
print(
    "113.0718:",
    bool(known_two_bond_row["up_to_two_113"])
)

print(
    "Known-structure up-to-two-bond score:",
    round(float(known_two_bond_row["up_to_two_bond_score"]), 6)
)

print("\nTop 5 candidates:")
print(
    two_bond_order[
        [
            "inchikey14",
            "up_to_two_bond_score",
            "mass_error_ppm"
        ]
    ].head(5).to_string(index=False)
)

CASMI 2026 - Two-Bond Ranking Diagnostic
----------------------------------------
Development molecule: m_145679
Candidates evaluated: 181
Mass-only known-structure rank: 75
One-bond known-structure rank: 20
Up-to-two-bond known-structure rank: 8

Candidates matching each peak:
86: one bond=19, up to two bonds=61
81: one bond=6, up to two bonds=51
113: one bond=10, up to two bonds=34

Known structure matches:
86.0603: True
81.0336: True
113.0718: True
Known-structure up-to-two-bond score: 1.76725

Top 5 candidates:
    inchikey14  up_to_two_bond_score  mass_error_ppm
ARBZLNAPSFGXRM               1.76725        0.681081
BGPFMZUGBBJXPN               1.76725        0.681081
HQTKDTDJIFJAMN               1.76725        0.681081
KTEULZXQURZPSN               1.76725        0.681081
NQYKAVYTROIOCS               1.76725        0.681081


In [39]:
# CASMI 2026 - Diagnose ties in two-bond fragment ranking

import numpy as np

known_row = combined_ranking.loc[
    combined_ranking["inchikey14"] == correct_key
].iloc[0]

known_score = float(known_row["up_to_two_bond_score"])
known_mass_error = float(known_row["mass_error_ppm"])

# Allow a tiny tolerance for floating-point comparison.
same_score = np.isclose(
    combined_ranking["up_to_two_bond_score"].to_numpy(),
    known_score,
    rtol=0,
    atol=1e-10
)

tied_candidates = combined_ranking.loc[same_score].copy()

higher_score_count = int(
    (
        combined_ranking["up_to_two_bond_score"]
        > known_score + 1e-10
    ).sum()
)

tied_with_lower_mass_error = int(
    (
        tied_candidates["mass_error_ppm"]
        < known_mass_error - 1e-10
    ).sum()
)

all_three_matches = (
    combined_ranking[
        ["up_to_two_86", "up_to_two_81", "up_to_two_113"]
    ]
    .all(axis=1)
)

print("CASMI 2026 - Two-Bond Score Tie Diagnostic")
print("------------------------------------------")
print("Development molecule:", development_id)
print("Candidates evaluated:", len(combined_ranking))
print("Known-structure score:", round(known_score, 6))
print("Known-structure final rank:", two_bond_rank)

print(
    "Candidates with strictly higher fragment score:",
    higher_score_count
)
print(
    "Candidates tied with known-structure score:",
    len(tied_candidates)
)
print(
    "Candidates matching all three peaks:",
    int(all_three_matches.sum())
)
print(
    "Tied candidates with smaller mass error:",
    tied_with_lower_mass_error
)

print("\nKnown structure among tied candidates:")
print(
    tied_candidates.sort_values(
        ["mass_error_ppm", "inchikey14"]
    )
    .reset_index(drop=True)
    .assign(tie_position=lambda df: np.arange(1, len(df) + 1))
    .loc[
        lambda df: df["inchikey14"] == correct_key,
        ["inchikey14", "tie_position", "mass_error_ppm"]
    ]
    .to_string(index=False)
)

CASMI 2026 - Two-Bond Score Tie Diagnostic
------------------------------------------
Development molecule: m_145679
Candidates evaluated: 181
Known-structure score: 1.76725
Known-structure final rank: 8
Candidates with strictly higher fragment score: 0
Candidates tied with known-structure score: 10
Candidates matching all three peaks: 10
Tied candidates with smaller mass error: 0

Known structure among tied candidates:
    inchikey14  tie_position  mass_error_ppm
XHTLRVHKUGTNCU             8        0.681081


In [40]:
# CASMI 2026 - Prepare a multi-molecule development experiment

import numpy as np
import pandas as pd

# Use only the original 20-molecule development subset.
# Select molecules using adduct and candidate count, not their labels.
eligible = []

for molecule_id in evaluation_ids:
    protonated_spectra = evaluation_queries.loc[
        (evaluation_queries["molecule_id"] == molecule_id)
        & (evaluation_queries["adduct"] == "[M+H]+")
    ]

    candidate_count = len(evaluation_candidates[molecule_id])

    if (
        molecule_id != development_id
        and not protonated_spectra.empty
        and 25 <= candidate_count <= 250
    ):
        eligible.append({
            "molecule_id": molecule_id,
            "candidate_count": candidate_count,
            "protonated_spectra": len(protonated_spectra)
        })

eligible_df = pd.DataFrame(eligible)

assert len(eligible_df) >= 4, (
    "Fewer than four additional development molecules meet "
    "the current selection criteria."
)

rng = np.random.default_rng(2026)

additional_ids = rng.choice(
    eligible_df["molecule_id"].to_numpy(),
    size=4,
    replace=False
).tolist()

fragment_development_ids = [development_id] + additional_ids

experiment_plan = pd.DataFrame([
    {
        "molecule_id": molecule_id,
        "candidate_count": len(evaluation_candidates[molecule_id]),
        "protonated_spectra": int(
            (
                (evaluation_queries["molecule_id"] == molecule_id)
                & (evaluation_queries["adduct"] == "[M+H]+")
            ).sum()
        )
    }
    for molecule_id in fragment_development_ids
])

print("CASMI 2026 - Fragment Development Experiment")
print("--------------------------------------------")
print("Molecules selected:", len(fragment_development_ids))
print("Holdout molecules included:", len(
    set(fragment_development_ids) & set(holdout_ids)
))

print("\nSelected molecules:")
print(experiment_plan.to_string(index=False))

assert not (
    set(fragment_development_ids) & set(holdout_ids)
)

print("\nDevelopment subset prepared successfully!")

CASMI 2026 - Fragment Development Experiment
--------------------------------------------
Molecules selected: 5
Holdout molecules included: 0

Selected molecules:
molecule_id  candidate_count  protonated_spectra
   m_145679              181                   3
   m_c886af              124                   1
   m_15dab4               51                   3
   m_dc7be7              212                   3
   m_30024e               66                   3

Development subset prepared successfully!


In [41]:
# CASMI 2026 - Generic fragment matching from observed peak masses

from itertools import combinations
from collections import Counter
from rdkit import Chem
import numpy as np
import pandas as pd

ELECTRON_MASS_DA = 0.000548579909065

FRAGMENT_ATOMIC_MASSES = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}


def match_observed_fragments(
    smiles,
    observed_mzs,
    observed_intensities,
    ppm_tolerance=20.0,
    max_bond_cuts=2,
    max_hydrogen_shift=2
):
    """
    Approximate structural matching for positive-mode fragment ions.

    Returns a boolean match for each observed peak.
    A peak is counted once, even if multiple bond cuts explain it.
    """

    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        return None

    mzs = np.asarray(observed_mzs, dtype=float)
    intensities = np.asarray(observed_intensities, dtype=float)

    assert mzs.shape == intensities.shape
    assert np.all(np.isfinite(mzs))
    assert np.all(mzs > 0)

    molecule_with_h = Chem.AddHs(molecule)

    atom_symbols = [
        atom.GetSymbol()
        for atom in molecule_with_h.GetAtoms()
    ]

    if not set(atom_symbols).issubset(FRAGMENT_ATOMIC_MASSES):
        return None

    bonds = [
        (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        for bond in molecule.GetBonds()
    ]

    matched = np.zeros(len(mzs), dtype=bool)
    minimum_bond_cuts = np.zeros(len(mzs), dtype=np.int8)

    for number_of_cuts in range(1, max_bond_cuts + 1):

        for bonds_to_cut in combinations(bonds, number_of_cuts):

            editable = Chem.RWMol(molecule_with_h)

            for atom_a, atom_b in bonds_to_cut:
                editable.RemoveBond(atom_a, atom_b)

            fragments = Chem.GetMolFrags(
                editable.GetMol(),
                asMols=False
            )

            if len(fragments) < 2:
                continue

            for atom_indices in fragments:

                counts = Counter(
                    atom_symbols[index]
                    for index in atom_indices
                )

                heavy_atom_count = sum(
                    count
                    for element, count in counts.items()
                    if element != "H"
                )

                if heavy_atom_count == 0:
                    continue

                base_mass = sum(
                    FRAGMENT_ATOMIC_MASSES[element] * count
                    for element, count in counts.items()
                )

                # Approximate hydrogen rearrangements.
                # These are candidate compositions, not proven
                # fragmentation mechanisms.
                for hydrogen_shift in range(
                    -max_hydrogen_shift,
                    max_hydrogen_shift + 1
                ):

                    if counts["H"] + hydrogen_shift < 0:
                        continue

                    theoretical_mz = (
                        base_mass
                        + hydrogen_shift
                        * FRAGMENT_ATOMIC_MASSES["H"]
                        - ELECTRON_MASS_DA
                    )

                    if theoretical_mz <= 0:
                        continue

                    ppm_errors = (
                        np.abs(mzs - theoretical_mz)
                        / theoretical_mz
                        * 1_000_000
                    )

                    new_matches = (
                        (ppm_errors <= ppm_tolerance)
                        & ~matched
                    )

                    minimum_bond_cuts[new_matches] = number_of_cuts
                    matched |= new_matches

            if matched.all():
                break

        if matched.all():
            break

    return {
        "matched_peaks": int(matched.sum()),
        "total_peaks": len(mzs),
        "matched_intensity": float(intensities[matched].sum()),
        "one_bond_matches": int(
            (minimum_bond_cuts == 1).sum()
        ),
        "two_bond_matches": int(
            (minimum_bond_cuts == 2).sum()
        )
    }


# Small smoke test: one mass-ranked candidate per development molecule.
# No correct-structure labels are used here.
test_rows = []

for molecule_id in fragment_development_ids:

    query_rows = evaluation_queries.loc[
        (evaluation_queries["molecule_id"] == molecule_id)
        & (evaluation_queries["adduct"] == "[M+H]+")
    ].sort_values("spectrum_id")

    selected_spectrum = query_rows.iloc[0]

    mzs = np.asarray(
        selected_spectrum["ms2_mzs"],
        dtype=float
    )

    intensities = np.asarray(
        selected_spectrum["ms2_normalized_intensities"],
        dtype=float
    )

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    strongest = np.argsort(-intensities)[:5]

    query_mass = float(mass_lookup.loc[molecule_id])

    candidates = library_with_mass.loc[
        library_with_mass["inchikey14"].isin(
            evaluation_candidates[molecule_id]
        ),
        ["inchikey14", "normalized_smiles", "reference_mass"]
    ].copy()

    candidates["mass_error"] = (
        candidates["reference_mass"] - query_mass
    ).abs()

    # Select by mass only, without looking up the correct label.
    candidate = candidates.sort_values(
        ["mass_error", "inchikey14"]
    ).iloc[0]

    result = match_observed_fragments(
        candidate["normalized_smiles"],
        mzs[strongest],
        intensities[strongest]
    )

    test_rows.append({
        "molecule_id": molecule_id,
        "candidate_key": candidate["inchikey14"],
        **result
    })


generic_fragment_smoke_test = pd.DataFrame(test_rows)

print("CASMI 2026 - Generic Fragment Matching")
print("--------------------------------------")
print("Development molecules tested:", len(test_rows))
print("Holdout labels used: 0")

print("\nOne mass-ranked candidate per molecule:")
print(
    generic_fragment_smoke_test.to_string(index=False)
)

print("\nGeneric fragment-matching function ready!")

CASMI 2026 - Generic Fragment Matching
--------------------------------------
Development molecules tested: 5
Holdout labels used: 0

One mass-ranked candidate per molecule:
molecule_id  candidate_key  matched_peaks  total_peaks  matched_intensity  one_bond_matches  two_bond_matches
   m_145679 AAPOCPWDQZSQRR              2            5           1.318452                 1                 1
   m_c886af BDNWJTRKUBXAGH              0            5           0.000000                 0                 0
   m_15dab4 VKTHZNIOCQUFCN              0            5           0.000000                 0                 0
   m_dc7be7 BFLMJFPEGXFCNQ              0            5           0.000000                 0                 0
   m_30024e AQYFXRPHXFRMDI              1            5           0.131279                 0                 1

Generic fragment-matching function ready!


In [42]:
# CASMI 2026 - Check fragment coverage of known development structures

import numpy as np
import pandas as pd

known_structure_rows = []

for molecule_id in fragment_development_ids:

    # Use the same spectrum-selection procedure as the smoke test.
    query_rows = evaluation_queries.loc[
        (evaluation_queries["molecule_id"] == molecule_id)
        & (evaluation_queries["adduct"] == "[M+H]+")
    ].sort_values("spectrum_id")

    selected_spectrum = query_rows.iloc[0]

    mzs = np.asarray(
        selected_spectrum["ms2_mzs"], dtype=float
    )

    intensities = np.asarray(
        selected_spectrum["ms2_normalized_intensities"],
        dtype=float
    )

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    strongest = np.argsort(-intensities)[:5]

    # Labels are used ONLY to diagnose the known structure.
    known_key = labels.loc[molecule_id]

    known_smiles = library_with_mass.loc[
        library_with_mass["inchikey14"] == known_key,
        "normalized_smiles"
    ].iloc[0]

    result = match_observed_fragments(
        known_smiles,
        mzs[strongest],
        intensities[strongest]
    )

    known_structure_rows.append({
        "molecule_id": molecule_id,
        "known_structure_key": known_key,
        **result
    })

known_fragment_coverage = pd.DataFrame(
    known_structure_rows
)

print("CASMI 2026 - Known-Structure Fragment Coverage")
print("----------------------------------------------")
print("Development molecules:", len(known_fragment_coverage))
print("Holdout molecules used: 0")

print("\nFragment matches for the known structures:")
print(
    known_fragment_coverage[
        [
            "molecule_id",
            "matched_peaks",
            "total_peaks",
            "matched_intensity",
            "one_bond_matches",
            "two_bond_matches"
        ]
    ].to_string(index=False)
)

print(
    "\nKnown structures with at least one matched peak:",
    int((known_fragment_coverage["matched_peaks"] > 0).sum())
)

CASMI 2026 - Known-Structure Fragment Coverage
----------------------------------------------
Development molecules: 5
Holdout molecules used: 0

Fragment matches for the known structures:
molecule_id  matched_peaks  total_peaks  matched_intensity  one_bond_matches  two_bond_matches
   m_145679              3            5           1.767250                 2                 1
   m_c886af              4            5           2.348439                 0                 4
   m_15dab4              2            5           1.155015                 1                 1
   m_dc7be7              2            5           0.874648                 1                 1
   m_30024e              3            5           0.899894                 1                 2

Known structures with at least one matched peak: 5


In [43]:
# CASMI 2026 - Rank all candidates for one development molecule

import time
import numpy as np
import pandas as pd

target_id = "m_c886af"

# Select the same [M+H]+ spectrum used in the smoke test.
query_row = (
    evaluation_queries.loc[
        (evaluation_queries["molecule_id"] == target_id)
        & (evaluation_queries["adduct"] == "[M+H]+")
    ]
    .sort_values("spectrum_id")
    .iloc[0]
)

mzs = np.asarray(query_row["ms2_mzs"], dtype=float)
intensities = np.asarray(
    query_row["ms2_normalized_intensities"], dtype=float
)

valid = (
    np.isfinite(mzs)
    & np.isfinite(intensities)
    & (mzs > 0)
    & (intensities > 0)
)

mzs = mzs[valid]
intensities = intensities[valid]

strongest = np.argsort(-intensities)[:5]

# Candidates are selected by precursor mass, without using labels.
query_mass = float(mass_lookup.loc[target_id])

candidates = (
    library_with_mass.loc[
        library_with_mass["inchikey14"].isin(
            evaluation_candidates[target_id]
        ),
        ["inchikey14", "normalized_smiles", "reference_mass"]
    ]
    .drop_duplicates("inchikey14")
    .copy()
)

candidates["mass_error_ppm"] = (
    np.abs(candidates["reference_mass"] - query_mass)
    / query_mass * 1_000_000
)

ranking_rows = []
start_time = time.perf_counter()

for index, row in enumerate(
    candidates.itertuples(index=False), start=1
):
    result = match_observed_fragments(
        row.normalized_smiles,
        mzs[strongest],
        intensities[strongest]
    )

    ranking_rows.append({
        "inchikey14": row.inchikey14,
        "fragment_score": (
            result["matched_intensity"]
            if result is not None
            else np.nan
        ),
        "matched_peaks": (
            result["matched_peaks"]
            if result is not None
            else 0
        ),
        "mass_error_ppm": row.mass_error_ppm
    })

    if index % 25 == 0 or index == len(candidates):
        print(f"Candidates processed: {index}/{len(candidates)}")

fragment_ranking_df = pd.DataFrame(ranking_rows)

mass_order = fragment_ranking_df.sort_values(
    ["mass_error_ppm", "inchikey14"]
).reset_index(drop=True)

fragment_order = fragment_ranking_df.sort_values(
    ["fragment_score", "mass_error_ppm", "inchikey14"],
    ascending=[False, True, True],
    na_position="last"
).reset_index(drop=True)

# Use the known label only AFTER both rankings are complete.
known_key = labels.loc[target_id]

def find_rank(ordered, key):
    positions = np.flatnonzero(
        ordered["inchikey14"].to_numpy() == key
    )
    return int(positions[0]) + 1 if len(positions) == 1 else None

mass_rank = find_rank(mass_order, known_key)
fragment_rank = find_rank(fragment_order, known_key)

known_score = fragment_ranking_df.loc[
    fragment_ranking_df["inchikey14"] == known_key,
    "fragment_score"
].iloc[0]

tied_count = int(
    np.isclose(
        fragment_ranking_df["fragment_score"].to_numpy(
            dtype=float
        ),
        known_score,
        rtol=0,
        atol=1e-10
    ).sum()
)

print("\nCASMI 2026 - Fragment Ranking: m_c886af")
print("----------------------------------------")
print("Candidates evaluated:", len(fragment_ranking_df))
print("Time (seconds):", round(time.perf_counter() - start_time, 1))
print("Mass-only known-structure rank:", mass_rank)
print("Fragment known-structure rank:", fragment_rank)
print("Known-structure fragment score:", round(float(known_score), 6))
print("Candidates tied with known score:", tied_count)
print("Known structure in fragment Top 25:", fragment_rank <= 25)

print("\nTop 5 candidates by fragment score:")
print(
    fragment_order[
        ["inchikey14", "fragment_score",
         "matched_peaks", "mass_error_ppm"]
    ].head(5).to_string(index=False)
)

Candidates processed: 25/124
Candidates processed: 50/124
Candidates processed: 75/124
Candidates processed: 100/124
Candidates processed: 124/124

CASMI 2026 - Fragment Ranking: m_c886af
----------------------------------------
Candidates evaluated: 124
Time (seconds): 5.7
Mass-only known-structure rank: 3
Fragment known-structure rank: 3
Known-structure fragment score: 2.348439
Candidates tied with known score: 2
Known structure in fragment Top 25: True

Top 5 candidates by fragment score:
    inchikey14  fragment_score  matched_peaks  mass_error_ppm
XCTQUXRJZMVEPT        2.923587              5       19.979533
TVWHFGAXRQIXET        2.588736              4       19.979533
YDWBNVYNVLNQEF        2.348439              4        1.264740
PAKPRKAXESAUKW        2.348439              4        3.908264
LARMPXBQYOHJTC        1.588736              3       15.156423


In [44]:
# CASMI 2026 - Evaluate fragment ranking on remaining development molecules

import time
import numpy as np
import pandas as pd

remaining_ids = [
    molecule_id
    for molecule_id in fragment_development_ids
    if molecule_id != "m_c886af"
]

development_ranking_rows = []

print("CASMI 2026 - Multi-Molecule Fragment Ranking")
print("---------------------------------------------")

for molecule_id in remaining_ids:

    # Use the first [M+H]+ spectrum, as in the previous smoke test.
    query_row = (
        evaluation_queries.loc[
            (evaluation_queries["molecule_id"] == molecule_id)
            & (evaluation_queries["adduct"] == "[M+H]+")
        ]
        .sort_values("spectrum_id")
        .iloc[0]
    )

    mzs = np.asarray(query_row["ms2_mzs"], dtype=float)
    intensities = np.asarray(
        query_row["ms2_normalized_intensities"],
        dtype=float
    )

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    strongest = np.argsort(-intensities)[:5]
    query_mass = float(mass_lookup.loc[molecule_id])

    candidates = (
        library_with_mass.loc[
            library_with_mass["inchikey14"].isin(
                evaluation_candidates[molecule_id]
            ),
            ["inchikey14", "normalized_smiles", "reference_mass"]
        ]
        .drop_duplicates("inchikey14")
        .copy()
    )

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    ranking_rows = []
    start_time = time.perf_counter()

    for row in candidates.itertuples(index=False):

        result = match_observed_fragments(
            row.normalized_smiles,
            mzs[strongest],
            intensities[strongest]
        )

        ranking_rows.append({
            "inchikey14": row.inchikey14,
            "fragment_score": (
                result["matched_intensity"]
                if result is not None
                else np.nan
            ),
            "matched_peaks": (
                result["matched_peaks"]
                if result is not None
                else 0
            ),
            "mass_error_ppm": row.mass_error_ppm
        })

    scores = pd.DataFrame(ranking_rows)

    mass_order = scores.sort_values(
        ["mass_error_ppm", "inchikey14"]
    ).reset_index(drop=True)

    fragment_order = scores.sort_values(
        ["fragment_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    # Use the label only AFTER both rankings have been generated.
    known_key = labels.loc[molecule_id]

    mass_rank = find_rank(mass_order, known_key)
    fragment_rank = find_rank(fragment_order, known_key)

    known_score = float(
        scores.loc[
            scores["inchikey14"] == known_key,
            "fragment_score"
        ].iloc[0]
    )

    tied_count = int(
        np.isclose(
            scores["fragment_score"].to_numpy(dtype=float),
            known_score,
            rtol=0,
            atol=1e-10
        ).sum()
    )

    development_ranking_rows.append({
        "molecule_id": molecule_id,
        "candidates": len(scores),
        "mass_rank": mass_rank,
        "fragment_rank": fragment_rank,
        "known_fragment_score": round(known_score, 6),
        "candidates_tied_with_known": tied_count,
        "runtime_seconds": round(
            time.perf_counter() - start_time, 1
        )
    })

    print(
        f"Completed {molecule_id}: "
        f"{len(scores)} candidates"
    )

remaining_development_results = pd.DataFrame(
    development_ranking_rows
)

print("\nCASMI 2026 - Remaining Development Results")
print("------------------------------------------")
print(
    remaining_development_results.to_string(index=False)
)

CASMI 2026 - Multi-Molecule Fragment Ranking
---------------------------------------------
Completed m_145679: 181 candidates
Completed m_15dab4: 51 candidates
Completed m_dc7be7: 212 candidates
Completed m_30024e: 66 candidates

CASMI 2026 - Remaining Development Results
------------------------------------------
molecule_id  candidates  mass_rank  fragment_rank  known_fragment_score  candidates_tied_with_known  runtime_seconds
   m_145679         181         75              8              1.767250                           9              8.9
   m_15dab4          51         23              2              1.155015                           2              2.8
   m_dc7be7         212         13             56              0.874648                           3             12.0
   m_30024e          66         26              9              0.899894                          16              3.2


In [45]:
# CASMI 2026 - Test the unchanged fragment method on more development molecules

import time
import numpy as np
import pandas as pd

new_development_ids = [
    molecule_id
    for molecule_id in evaluation_ids
    if molecule_id not in fragment_development_ids
]

additional_results = []
skipped_molecules = []

print("CASMI 2026 - Additional Fragment Evaluation")
print("-------------------------------------------")

for molecule_id in new_development_ids:

    query_rows = evaluation_queries.loc[
        (evaluation_queries["molecule_id"] == molecule_id)
        & (evaluation_queries["adduct"] == "[M+H]+")
    ].sort_values("spectrum_id")

    candidate_keys = evaluation_candidates[molecule_id]

    # Keep the experiment bounded in execution time.
    if query_rows.empty or len(candidate_keys) > 250:
        skipped_molecules.append({
            "molecule_id": molecule_id,
            "reason": (
                "No [M+H]+ spectrum"
                if query_rows.empty
                else "More than 250 candidates"
            )
        })
        continue

    query_row = query_rows.iloc[0]

    mzs = np.asarray(query_row["ms2_mzs"], dtype=float)
    intensities = np.asarray(
        query_row["ms2_normalized_intensities"],
        dtype=float
    )

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs > 0)
        & (intensities > 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    if len(mzs) == 0:
        skipped_molecules.append({
            "molecule_id": molecule_id,
            "reason": "No valid fragment peaks"
        })
        continue

    strongest = np.argsort(-intensities)[:5]
    query_mass = float(mass_lookup.loc[molecule_id])

    candidates = (
        library_with_mass.loc[
            library_with_mass["inchikey14"].isin(candidate_keys),
            ["inchikey14", "normalized_smiles", "reference_mass"]
        ]
        .drop_duplicates("inchikey14")
        .copy()
    )

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    rows = []
    start_time = time.perf_counter()

    for candidate in candidates.itertuples(index=False):

        result = match_observed_fragments(
            candidate.normalized_smiles,
            mzs[strongest],
            intensities[strongest]
        )

        rows.append({
            "inchikey14": candidate.inchikey14,
            "mass_error_ppm": candidate.mass_error_ppm,
            "fragment_score": (
                result["matched_intensity"]
                if result is not None
                else np.nan
            )
        })

    scores = pd.DataFrame(rows)

    mass_order = scores.sort_values(
        ["mass_error_ppm", "inchikey14"]
    ).reset_index(drop=True)

    fragment_order = scores.sort_values(
        ["fragment_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    # Labels are used only after both rankings are complete.
    known_key = labels.loc[molecule_id]

    additional_results.append({
        "molecule_id": molecule_id,
        "candidates": len(scores),
        "mass_rank": find_rank(mass_order, known_key),
        "fragment_rank": find_rank(fragment_order, known_key),
        "runtime_seconds": round(
            time.perf_counter() - start_time, 1
        )
    })

    print(
        f"Completed {molecule_id}: "
        f"{len(scores)} candidates"
    )


additional_development_results = pd.DataFrame(
    additional_results
)

print("\nCASMI 2026 - Additional Development Results")
print("-------------------------------------------")

if not additional_development_results.empty:
    print(
        additional_development_results.to_string(index=False)
    )

    print(
        "\nMass-only Top 25:",
        int(
            (additional_development_results["mass_rank"] <= 25)
            .sum()
        ),
        "/",
        len(additional_development_results)
    )

    print(
        "Fragment Top 25:",
        int(
            (additional_development_results["fragment_rank"] <= 25)
            .sum()
        ),
        "/",
        len(additional_development_results)
    )

print("\nSkipped molecules:", len(skipped_molecules))

if skipped_molecules:
    print(pd.DataFrame(skipped_molecules).to_string(index=False))

CASMI 2026 - Additional Fragment Evaluation
-------------------------------------------
Completed m_bf6cfe: 102 candidates
Completed m_f866a0: 67 candidates
Completed m_e138be: 27 candidates
Completed m_c9a4bc: 107 candidates
Completed m_1de28b: 92 candidates
Completed m_153108: 22 candidates
Completed m_7108be: 33 candidates
Completed m_7b993b: 230 candidates
Completed m_c3e52e: 124 candidates
Completed m_d07b86: 123 candidates
Completed m_741f2c: 72 candidates
Completed m_b8f328: 49 candidates
Completed m_8d75c7: 9 candidates

CASMI 2026 - Additional Development Results
-------------------------------------------
molecule_id  candidates  mass_rank  fragment_rank  runtime_seconds
   m_bf6cfe         102         54              6              3.7
   m_f866a0          67          7              1              4.7
   m_e138be          27         18              4              1.0
   m_c9a4bc         107         25              6              4.5
   m_1de28b          92         28        